# 02 · Burnt area: what Sentinel-2 saw that fire counts missed

**Project:** Paddy Burnt-Area Intelligence (Satellite Mission Intelligence for India)
**Question for this notebook:** in Sangrur district (pre-2021 boundary), how much harvested paddy burned in the 2025 season, when did it burn, and what share of that burnt area was never flagged by a VIIRS fire detection?

**How it works**
1. Copernicus's openEO service builds cloud-masked 5-day Sentinel-2 composites on its own servers: red, near infrared and two short-wave infrared bands, at 20 m. Only the finished composites are downloaded.
2. On your computer, each pixel's season is traced: green paddy, then harvested, then burnt or not, using three indices (NDVI, NBR and MIRBI).
3. Each burnt pixel is checked against VIIRS and MODIS fire detections from nearby, on the days it could have burned.
4. For checking the map, random validation points get image strips to label by eye, made from single Sentinel-2 dates at 10 m.

**Run it in steps, because openEO jobs use credits** (Copernicus gives 10,000 free a month)
1. **Pilot.** Click **Run All** as it is. Two small jobs run on a 10 km × 10 km box: the composites, and a check of how the cloud mask treats burnt fields. The notebook then projects the cost and download size of the whole district, and every later step runs on the pilot box so you can check it cheaply.
2. **Whole district.** If the projections fit, set `RUN_FULL = True` in Settings and click **Run All** again. The composites job runs on Copernicus's servers; on 26 Oct 2026 it took 11 minutes and 8 credits. A last, small job then fetches single dates around the validation points for their image strips. The downloads go to `data/raw/sentinel2/`, which git ignores.
3. **Label the validation points** (section 10), then **Run All** once more: section 11 checks the burn rule against the labels and reports its accuracy.

The first time, openEO prints a login link: open it and sign in with your Copernicus account. If the notebook stops while a job is running, just run it again. It picks up the same job instead of starting (and paying for) a new one.

**What it produces**

| Output | Where |
|---|---|
| Classes (unconfirmed late burns are a class of their own), burn window, harvest window and last clear view before the burn (20 m GeoTIFF) | `data/processed/s2_burn_classes_<area>_2025.tif` |
| Headline numbers, including the missed share | `data/processed/s2_burn_summary_<area>_2025.csv` |
| Harvest, burning and fire detections by 5-day window | `data/processed/s2_burn_by_window_<area>_2025.csv` |
| 1 km grid for the dashboard's two tabs | `data/processed/s2_burn_grid_1km_<area>_2025.geojson` |
| Fire detections used for matching | `data/processed/firms_detections_<area>_2025.csv` |
| Cloud-mask check | `data/processed/s2_scl_check_pilot_2025.csv` |
| Validation points and label sheet (whole district only) | `data/processed/validation_*` |
| Image strips for labelling: single dates (`P001_dates.png`) and 5-day composites (`P001.png`) | `data/raw/validation_chips/` |
| Charts and maps | `maps/notebook02/*.png` |
| Source register rows | `sources/source_register.csv` |

`<area>` is `pilot` until the whole-district run, then `sangrur`.

**Other seasons.** Set `SEASON_YEAR` in Settings (for example to 2021) to map another season with the same rule. That season gets its own download folder (`data/raw/sentinel2/sangrur_2021/`), its own figure folder (`maps/notebook02/2021/`) and its own output files (`…_sangrur_2021…`), so the 2025 results stay as they are. The validation sections (10 and 11) run for 2025 only, because the labels belong to that season; another season's map is not checked against labels. Its pilot and cloud-mask check run again, because the scaling of the values can differ between years.

**The burn rule was checked at Checkpoint 3 (30 Oct 2026).** Thresholds fitted on half the labelled points did worse on the other half than the starting thresholds, so the starting ones stay. Burns first seen from 19 Nov are mapped as *unconfirmed* and kept out of the headline figures: none of the labelled points the rule calls burnt that late was a confirmed burn, and VIIRS saw far fewer fires per mapped hectare than earlier in the season. Damp soil under young wheat can pass all three burn tests. See the decision log.

In [ ]:
# Colab: add libraries its default image may lack. On your own computer, install requirements.txt into the
# project's environment first; there this cell only checks that the install worked.
import importlib.util, subprocess, sys

try:
    import google.colab  # noqa: F401  (this module exists only on Colab)
    ON_COLAB = True
except ImportError:
    ON_COLAB = False

NEEDED = ('openeo', 'rasterio', 'geopandas', 'scipy', 'matplotlib', 'requests', 'xarray', 'netCDF4')
if ON_COLAB:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'openeo', 'rasterio', 'geopandas', 'netCDF4'],
                   check=True)
else:
    missing = [name for name in NEEDED if importlib.util.find_spec(name) is None]
    if missing:
        raise ModuleNotFoundError('Missing: ' + ', '.join(missing) + ". Install requirements.txt into the "
                                  "project's environment and re-run.")
    print('Libraries found.')

## 1. Settings

In [ ]:
import os, io, json, re, shutil, time, unicodedata
from pathlib import Path

import numpy as np
import pandas as pd
import requests
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from matplotlib.patches import Patch, Rectangle
import geopandas as gpd
import xarray as xr
import rasterio
from rasterio import features
from rasterio.windows import Window
from shapely.geometry import box
from IPython.display import display
import openeo
from openeo.rest import OpenEoApiError

TODAY = pd.Timestamp.today().normalize()
ACCESS_DATE = TODAY.strftime('%Y-%m-%d')

try:
    from google.colab import drive, userdata  # type: ignore
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    drive.mount('/content/drive')
    PROJECT = Path('/content/drive/MyDrive/India-Space-Mission-Intelligence')
elif os.environ.get('PROJECT_DIR'):
    PROJECT = Path(os.environ['PROJECT_DIR']).resolve()
else:
    # On your computer: walk up from the working folder to the project root (the folder with requirements.txt).
    here = Path.cwd().resolve()
    PROJECT = next((p for p in (here, *here.parents)
                    if (p / 'requirements.txt').exists() and (p / 'notebooks').is_dir()), here.parent)

RAW = PROJECT / 'data' / 'raw'
PROCESSED = PROJECT / 'data' / 'processed'
FIGS = PROJECT / 'maps' / 'notebook02'
SOURCES = PROJECT / 'sources'
CHIPS = RAW / 'validation_chips'
for folder in (PROCESSED, FIGS, SOURCES):
    folder.mkdir(parents=True, exist_ok=True)

plt.rcParams.update({'figure.dpi': 110, 'axes.spines.top': False, 'axes.spines.right': False,
                     'axes.grid': True, 'grid.alpha': 0.3, 'font.size': 10})
print('Project folder:', PROJECT)

# --- Study district (decision log, 22 Oct 2026) -----------------------------------------------
DISTRICT, STATE = 'Sangrur', 'Punjab'
DISTRICT_LABEL = 'Sangrur (pre-2021 boundary)'
STATE_BBOX = (73.8, 29.5, 77.0, 32.6)   # Punjab, to pick the right polygon if a district name repeats
UTM_EPSG = 32643                        # UTM zone 43N, metres

# --- Season and composites --------------------------------------------------------------------
SEASON_YEAR = 2025                      # the season to map; 2025 is the one with validation labels
VALIDATED_SEASON = 2025                 # the season the validation points and labels belong to
COMPOSITE_START = f'{SEASON_YEAR}-09-05'    # two windows before the season (15 Sep), to see the crop while green
COMPOSITE_END = f'{SEASON_YEAR}-12-05'      # exclusive
WINDOW_DAYS = 5
RESOLUTION_M = 20
BANDS = ['B04', 'B08', 'B11', 'B12']    # red, near infrared, short-wave infrared 1 and 2
SCL_KEEP = [2, 4, 5, 6, 7]              # scene classes kept: dark area, vegetation, bare soil, water, unclassified.
                                        # Fresh burn scars are dark and can land in the dark-area or water classes,
                                        # so only no-data, saturated, cloud shadow, cloud, cirrus and snow are masked.
                                        # Section 5 checks this choice on the pilot box.
MAX_CLOUD_COVER = 80                    # skip scenes whose cloud cover is above this (%)

# --- openEO jobs and credits ------------------------------------------------------------------
OPENEO_URL = 'openeo.dataspace.copernicus.eu'
PILOT_KM = 10
RUN_FULL = False                        # set True after checking the pilot's projected cost and size
FULL_BUDGET_CREDITS = 3000              # the whole-district run stops if its projected cost is higher
PILOT_COST_OVERRIDE = None              # credits the pilot used, if openEO doesn't report them (see the Web Editor)
JOB_OPTIONS = None                      # e.g. {'executor-memory': '4G'} if the whole-district job runs out of memory
# Single dates in the burning peak, for the cloud-mask check:
SCL_CHECK_START, SCL_CHECK_END = f'{SEASON_YEAR}-10-15', f'{SEASON_YEAR}-11-25'
SCL_CHECK_MAX_CLOUD = 30

# --- Reflectance scaling (detected from the pilot unless set here) ----------------------------
DN_OFFSET = None                        # 0 = values already harmonised; 1000 = raw processing-baseline-04 values

# --- Burn rule (checked against the validation labels at Checkpoint 3, 30 Oct 2026) ------------
GREEN_NDVI = 0.6                        # a pixel counts as green paddy once its NDVI has reached this
HARVEST_NDVI = 0.35                     # after that, NDVI below this means harvested
BURN_NBR_MAX = -0.05                    # a harvested pixel counts as burnt when NBR is at most this ...
BURN_MIRBI_MIN = 1.65                   # ... MIRBI is at least this ...
BURN_NIR_MAX = 0.13                     # ... and NIR reflectance is at most this, in the same window. Fresh char is
                                        # dark; bare soil after harvest has the same NBR and MIRBI but is brighter.
                                        # Without this condition the 26 Oct pilot marked 91% of harvested land burnt.
UNCONFIRMED_FROM = f'{SEASON_YEAR}-11-19'   # burns first seen in a window starting on or after this date are mapped as
                                        # unconfirmed and kept out of the headline figures (decision log, 30 Oct 2026):
                                        # damp soil under young wheat can pass all three tests. None keeps every burn.
RULE_STATUS = 'checked against the validation labels at Checkpoint 3 (30 Oct 2026); thresholds kept'

# --- Matching burnt pixels with fire detections -------------------------------------------------
VIIRS_RADII_M = (375, 750)              # strict: one VIIRS pixel; lenient: two, for geolocation error
MODIS_RADIUS_M = 1000                   # one MODIS pixel
FIRMS_CANDIDATES = {                    # standard processing first; near real time fills any dates it lacks
    'VIIRS_SNPP': ['VIIRS_SNPP_SP', 'VIIRS_SNPP_NRT'],
    'VIIRS_NOAA20': ['VIIRS_NOAA20_SP', 'VIIRS_NOAA20_NRT'],
    'VIIRS_NOAA21': ['VIIRS_NOAA21_NRT'],   # FIRMS has no standard-processing NOAA-21 product
    'MODIS': ['MODIS_SP', 'MODIS_NRT'],
}
FIRMS_FIRST_SEASON = {'VIIRS_NOAA21': 2023}   # NOAA-21 launched in Nov 2022: no fires for earlier seasons
DAY_RANGE = 5                           # days per FIRMS request (lowered automatically if FIRMS asks)
PAUSE_S = 1.0

# --- Validation sample (Checkpoint 3) ------------------------------------------------------------
SAMPLE_SIZES = {'burnt': 100, 'harvested, not burnt': 100, 'other': 40}   # points per mapped class
SAMPLE_SEED = 2025
CHIP_PX = 75                            # composite strips: 75 px x 20 m = 1.5 km
MIN_LABELS = 100                        # labelled points needed before section 11 checks the rule
DATE_STRIPS = True                      # also fetch single dates at 10 m around each point (one small openEO job)
# Dates in those strips (end exclusive):
DATE_STRIP_START, DATE_STRIP_END = f'{SEASON_YEAR}-09-20', f'{SEASON_YEAR}-12-05'
DATE_STRIP_MAX_CLOUD = 60               # skip scenes with more cloud than this (%); the point itself must be clear
DATE_STRIP_M = 640                      # each single-date panel is 640 m across (64 px of 10 m)

if SEASON_YEAR != VALIDATED_SEASON:     # another season: its own figure folder, so the 2025 charts stay as they are
    FIGS = FIGS / str(SEASON_YEAR)
    FIGS.mkdir(parents=True, exist_ok=True)


def read_env_file(path):
    # KEY=value lines from a .env file; blank lines and lines starting with # are skipped.
    values = {}
    if path.exists():
        for line in path.read_text(encoding='utf-8-sig').splitlines():
            line = line.strip()
            if line and not line.startswith('#') and '=' in line:
                name, value = line.split('=', 1)
                values[name.strip()] = value.strip().strip('"\'')
    return values


def get_map_key():
    key = None
    if IN_COLAB:
        try:
            key = userdata.get('FIRMS_MAP_KEY')
        except Exception:
            key = None
    key = key or os.environ.get('FIRMS_MAP_KEY') or read_env_file(PROJECT / '.env').get('FIRMS_MAP_KEY')
    if not key:
        raise RuntimeError('No FIRMS map key found. Put the line FIRMS_MAP_KEY=your_key in the .env file in '
                           f'{PROJECT} (or add a Colab secret named FIRMS_MAP_KEY), then re-run this cell.')
    return key.strip()


MAP_KEY = get_map_key()
print('Map key loaded (not printed).')

## 2. Helper functions

FIRMS downloads work as in Notebook 01: 5-day requests, cached, with the key masked in any error message. The openEO helpers build each job, run it (or resume it if it is already running) and record it in a `job.json` file next to its download. They download the job's files one at a time: a file already complete is kept, and a download that breaks off is tried again, so a rerun after a dropped connection fetches only what is missing. Composites are read one window at a time, so even the whole district fits in memory.

In [ ]:
FIRMS_AREA = 'https://firms.modaps.eosdis.nasa.gov/api/area/csv/{key}/{source}/{bbox}/{days}/{date}'
FIRMS_AVAILABILITY = 'https://firms.modaps.eosdis.nasa.gov/api/data_availability/csv/{key}/all'
REGISTER = SOURCES / 'source_register.csv'
REGISTER_COLUMNS = ['source_id', 'dataset_or_product', 'provider', 'url', 'date_accessed', 'geography',
                    'time_range', 'resolution', 'licence_or_terms', 'preprocessing', 'known_limitations', 'used_in']
NOTEBOOK = 'notebooks/02_eo_analysis_burnt_area.ipynb'


def http_get(url, timeout=120):
    return requests.get(url, timeout=timeout)


def safe(text):
    # Error text without the FIRMS key (requests puts the full URL, key included, in its errors).
    return str(text).replace(MAP_KEY, '***')[:300]


def register_source(**row):
    # Add or replace one row of the source register, keyed by source_id.
    row = {c: str(row.get(c, '')) for c in REGISTER_COLUMNS}
    reg = pd.read_csv(REGISTER, dtype=str).fillna('') if REGISTER.exists() else pd.DataFrame(columns=REGISTER_COLUMNS)
    reg = reg[reg['source_id'] != row['source_id']]
    pd.concat([reg, pd.DataFrame([row])], ignore_index=True)[REGISTER_COLUMNS].to_csv(REGISTER, index=False)


def plain_name(name):
    # Lower-case ASCII form of a place name, so 'Sangrūr' or 'SANGRUR' match 'Sangrur'.
    return unicodedata.normalize('NFKD', str(name)).encode('ascii', 'ignore').decode('ascii').strip().lower()


SLUG = plain_name(DISTRICT).replace(' ', '_')
S2_RAW = RAW / 'sentinel2' / (SLUG if SEASON_YEAR == VALIDATED_SEASON else f'{SLUG}_{SEASON_YEAR}')
FIRMS_RAW = RAW / 'firms_district' / SLUG


# --- FIRMS ---------------------------------------------------------------------------------------
class DayRangeTooLong(Exception):
    # FIRMS refused the number of days in one request; args[0] is the most it accepts.
    pass


class SourceUnavailable(Exception):
    # FIRMS refused this product or these dates (anything 'invalid' that isn't the key or the day range).
    pass


def fetch_firms(source, start, days, bbox):
    start_ts = pd.Timestamp(start)
    end_ts = start_ts + pd.Timedelta(days - 1, unit='D')
    cache = FIRMS_RAW / source / f'{start_ts:%Y-%m-%d}_{days}d.csv'
    if cache.exists():
        return pd.read_csv(cache)
    url = FIRMS_AREA.format(key=MAP_KEY, source=source, bbox=','.join(f'{v:.4f}' for v in bbox), days=days,
                            date=f'{start_ts:%Y-%m-%d}')
    waits, message = [60, 120, 300], ''
    for attempt in range(len(waits) + 1):
        try:
            r = http_get(url)
            text = r.text.strip()
            if r.status_code == 200 and (text == '' or text.startswith('latitude')):
                df = pd.read_csv(io.StringIO(text)) if text else pd.DataFrame(columns=['latitude', 'longitude'])
                if end_ts < TODAY - pd.Timedelta(2, unit='D'):  # cache only periods that are complete
                    cache.parent.mkdir(parents=True, exist_ok=True)
                    df.to_csv(cache, index=False)
                time.sleep(PAUSE_S)
                return df
            message = safe(text)
        except requests.RequestException as err:
            message = safe(err)
        limit = re.search(r'day range.*?\[1\s*\.\.\s*(\d+)\]', message, flags=re.I)
        if limit:
            raise DayRangeTooLong(int(limit.group(1)))
        if 'invalid' in message.lower():
            if 'key' in message.lower():
                raise RuntimeError(f'FIRMS rejected the request: {message}. Check FIRMS_MAP_KEY in .env.')
            raise SourceUnavailable(message)
        if attempt < len(waits):
            print(f'  {source} {start_ts:%Y-%m-%d}: {message!r}; retrying in {waits[attempt]} s')
            time.sleep(waits[attempt])
    raise RuntimeError(f'Could not download {source} from {start_ts:%Y-%m-%d}: {message}')


def download_firms(source, start, end, bbox):
    global DAY_RANGE
    frames, day, end = [], pd.Timestamp(start), pd.Timestamp(end)
    while day <= end:
        days = int(min(DAY_RANGE, (end - day).days + 1))
        try:
            chunk = fetch_firms(source, day, days, bbox)
        except DayRangeTooLong as err:
            accepted = err.args[0]
            if not 1 <= accepted < days:
                raise RuntimeError(f'FIRMS refused a {days}-day request but says it accepts up to {accepted} days.')
            DAY_RANGE = accepted
            print(f'  FIRMS accepts at most {accepted} days per request; continuing with {accepted}-day blocks.')
            continue
        if len(chunk):
            frames.append(chunk.assign(source=source))
        day += pd.Timedelta(days, unit='D')
    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()


def firms_availability():
    # First and last date each FIRMS product covers (columns data_id, min_date, max_date), or None.
    try:
        r = http_get(FIRMS_AVAILABILITY.format(key=MAP_KEY), timeout=60)
        if r.status_code == 200 and r.text.strip().lower().startswith('data_id'):
            table = pd.read_csv(io.StringIO(r.text.strip()))
            for col in ('min_date', 'max_date'):
                table[col] = pd.to_datetime(table[col])
            return table.set_index('data_id')
        print('  FIRMS availability lookup answered:', safe(r.text.strip()[:120]))
    except Exception as err:
        print('  FIRMS availability lookup failed:', safe(err))
    return None


def plan_firms(candidates, start, end, availability):
    # Which product to use for which days: the first candidate wherever it has data, the next one for the days
    # it lacks, and so on. Returns ([(source, first_day, last_day), ...], [(first_day, last_day) not covered]).
    todo, plan = [(pd.Timestamp(start), pd.Timestamp(end))], []
    for source in candidates:
        if source not in availability.index:
            continue
        first, last = availability.loc[source, 'min_date'], availability.loc[source, 'max_date']
        remaining = []
        for a, b in todo:
            lo, hi = max(a, first), min(b, last)
            if lo > hi:
                remaining.append((a, b))
                continue
            plan.append((source, lo, hi))
            if a < lo:
                remaining.append((a, lo - pd.Timedelta(1, unit='D')))
            if hi < b:
                remaining.append((hi + pd.Timedelta(1, unit='D'), b))
        todo = remaining
    return sorted(plan, key=lambda p: p[1]), todo


# --- Study area ------------------------------------------------------------------------------------
def load_district():
    path = RAW / 'boundaries' / 'geoBoundaries-IND-ADM2.geojson'
    if not path.exists():
        raise RuntimeError(f'{path} is missing. Run Notebook 01 first: it downloads the district boundaries.')
    adm2 = gpd.read_file(path).to_crs(4326)
    match = adm2[adm2['shapeName'].map(plain_name) == plain_name(DISTRICT)]
    if len(match) > 1:
        match = match[match.intersects(box(*STATE_BBOX))]
    if len(match) != 1:
        raise RuntimeError(f'Expected one {DISTRICT} polygon in {path.name}, found {len(match)}.')
    return match[['shapeName', 'geometry']].reset_index(drop=True)


# --- openEO ------------------------------------------------------------------------------------------
def window_starts():
    first, stop = pd.Timestamp(COMPOSITE_START), pd.Timestamp(COMPOSITE_END)
    starts, day = [], first
    while day + pd.Timedelta(WINDOW_DAYS, unit='D') <= stop:
        starts.append(day)
        day += pd.Timedelta(WINDOW_DAYS, unit='D')
    return starts


WINDOWS = window_starts()
LABELS = [f'{d:%Y-%m-%d}' for d in WINDOWS]
INTERVALS = [[f'{d:%Y-%m-%d}', f'{d + pd.Timedelta(WINDOW_DAYS, unit="D"):%Y-%m-%d}'] for d in WINDOWS]
_CON = None


def connect_openeo():
    con = openeo.connect(OPENEO_URL)
    con.authenticate_oidc()   # first time: prints a login link; later runs reuse the saved login
    return con


def get_con():
    global _CON
    if _CON is None:
        _CON = connect_openeo()
    return _CON


def extent(geom):
    west, south, east, north = geom.bounds
    return {'west': west, 'south': south, 'east': east, 'north': north}


def load_s2(con, geom, start, end, max_cloud):
    # Sentinel-2 L2A bands plus the scene classification (SCL), on a 20 m UTM grid.
    s2 = con.load_collection('SENTINEL2_L2A', spatial_extent=extent(geom), temporal_extent=[start, end],
                             bands=BANDS + ['SCL'], max_cloud_cover=max_cloud)
    # Nearest neighbour keeps the scene classes intact; 10 m bands are sampled to 20 m.
    return s2.resample_spatial(resolution=RESOLUTION_M, projection=UTM_EPSG, method='near')


def build_cube(con, geom):
    # Cloud-masked 5-day median composites of BANDS, clipped to `geom` (a WGS84 shapely polygon).
    s2 = load_s2(con, geom, COMPOSITE_START, COMPOSITE_END, MAX_CLOUD_COVER)
    scl = s2.band('SCL')
    keep = None
    for cls in SCL_KEEP:
        keep = (scl == cls) if keep is None else (keep | (scl == cls))
    cube = s2.filter_bands(BANDS).mask(~keep)
    cube = cube.aggregate_temporal(intervals=INTERVALS, reducer='median', labels=LABELS)   # labelled by window start
    cube = cube.mask_polygon(geom)
    # Store as 16-bit whole numbers, half the size of the default 32-bit floats. This is a workaround openEO's
    # developers describe (github.com/Open-EO/openeo-geotrellis-extensions/issues/873); section 5 prints the type.
    return cube.linear_scale_range(1, 65534, 1, 65534)


def build_scl_check(con, geom):
    # Single clear-ish dates with their scene classes, unmasked, for the cloud-mask check.
    return load_s2(con, geom, SCL_CHECK_START, SCL_CHECK_END, SCL_CHECK_MAX_CLOUD)


DATE_BANDS = ['B02', 'B03', 'B04', 'B08', 'B11', 'B12']   # blue, green, red, NIR, SWIR 1 and 2
DATE_STRIP_VERSION = '2'   # raised when the strips' drawing changes, so older strips are redrawn


def point_squares(pts_utm):
    # GeoJSON squares DATE_STRIP_M across around each point (UTM x, y), in WGS84, keyed by point_id.
    half = DATE_STRIP_M / 2
    squares = gpd.GeoSeries([box(x - half, y - half, x + half, y + half) for x, y in zip(pts_utm['x'], pts_utm['y'])],
                            crs=UTM_EPSG).to_crs(4326)
    return {'type': 'FeatureCollection',
            'features': [{'type': 'Feature', 'properties': {'point_id': pid}, 'geometry': geom.__geo_interface__}
                         for pid, geom in zip(pts_utm['point_id'], squares)]}


def build_date_cube(con, squares):
    # Every single date (no compositing, no cloud masking) at 10 m, cut to the squares around the points.
    # Saved as one netCDF file per square ("sample_by_feature"), so the download stays small.
    minx, miny, maxx, maxy = gpd.GeoDataFrame.from_features(squares['features'], crs=4326).total_bounds
    s2 = con.load_collection('SENTINEL2_L2A', spatial_extent={'west': minx, 'south': miny, 'east': maxx, 'north': maxy},
                             temporal_extent=[DATE_STRIP_START, DATE_STRIP_END], bands=DATE_BANDS + ['SCL'],
                             max_cloud_cover=DATE_STRIP_MAX_CLOUD)
    s2 = s2.resample_spatial(resolution=10, projection=UTM_EPSG, method='near')
    return s2.filter_spatial(squares).linear_scale_range(0, 65534, 0, 65534)


def load_state(folder):
    path = Path(folder) / 'job.json'
    return json.loads(path.read_text()) if path.exists() else {}


def save_state(folder, state):
    (Path(folder) / 'job.json').write_text(json.dumps(state, indent=2, default=str))


def job_status(job):
    # The job's status, tolerating short network drop-outs (or a laptop that went to sleep).
    for attempt in range(10):
        try:
            return job.status()
        except (requests.ConnectionError, requests.Timeout) as err:
            print(f'  Could not reach openEO ({str(err)[:100]}); trying again in a minute.')
            time.sleep(60)
    raise RuntimeError('Could not reach openEO for 10 minutes. Run the cell again later: it resumes the same job.')


def download_results(job, out_dir, attempts=5):
    # Download the job's files one at a time. A file already on disk at its full size is kept, and a download
    # that breaks off (a dropped connection) is tried again, so a rerun fetches only what is missing.
    assets = job.get_results().get_assets()
    kept = 0
    for asset in assets:
        name = getattr(asset, 'key', None) or asset.name
        target = Path(out_dir) / name
        for attempt in range(1, attempts + 1):
            try:
                with job.connection.head(asset.href, stream=True, check_error=False) as head:
                    size = int(head.headers['Content-Length']) if head.ok and 'Content-Length' in head.headers \
                        else None
                if size is not None and target.exists() and target.stat().st_size == size:
                    kept += 1
                    break
                asset.download(target)
                if size is not None and target.stat().st_size != size:
                    raise requests.ConnectionError('the file arrived incomplete')
                break
            except (requests.ConnectionError, requests.Timeout, requests.exceptions.ChunkedEncodingError) as err:
                target.unlink(missing_ok=True)
                if attempt == attempts:
                    raise RuntimeError(f'The download of {name} broke off {attempts} times ({type(err).__name__}). '
                                       'Run the cell again later: it resumes the same job and keeps the files '
                                       'already downloaded.') from err
                print(f'  The download of {name} broke off ({type(err).__name__}); trying again in a minute.')
                time.sleep(60)
    print(f'  Result files: {len(assets) - kept} downloaded, {kept} already complete.')


def run_batch(con, cube, out_dir, title, out_format='GTiff', **format_options):
    # Run a batch job, or resume the one recorded in out_dir/job.json, then download its files.
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    state = load_state(out_dir)
    if state.get('job_id'):
        job = con.job(state['job_id'])
        print(f'Resuming job {state["job_id"]}.')
        try:
            job.describe()
        except OpenEoApiError as err:
            raise RuntimeError(f'openEO no longer has job {state["job_id"]} ({str(err)[:100]}). Delete '
                               f'{out_dir / "job.json"} to start a fresh job.') from err
    else:
        job = cube.create_job(out_format=out_format, title=title, job_options=JOB_OPTIONS, **format_options)
        state = {'job_id': job.job_id, 'title': title, 'created': pd.Timestamp.now().isoformat(timespec='seconds')}
        save_state(out_dir, state)
    status = job_status(job)
    if status == 'created':
        job.start()
        print(f'Started job {job.job_id} on the Copernicus servers. This cell waits and checks every minute.')
        status = job_status(job)
    started, last_shown = time.time(), None
    while status in ('created', 'queued', 'running'):
        if status != last_shown:
            print(f'  {pd.Timestamp.now():%H:%M} {status}')
            last_shown = status
        time.sleep(60 if status == 'running' else 20)
        status = job_status(job)
    print(f'  {pd.Timestamp.now():%H:%M} {status} after {(time.time() - started) / 60:.0f} min')
    if status != 'finished':
        errors = [entry.get('message', '') for entry in job.logs(level='error')][-5:]
        state['status'] = status
        save_state(out_dir, state)
        raise RuntimeError(f'Job {job.job_id} ended with status "{status}". Last errors: {errors}. '
                           f'Once the cause is fixed, delete {out_dir / "job.json"} to start a fresh job.')
    download_results(job, out_dir)
    info = job.describe()
    state.update(status='finished', costs=info.get('costs'), usage=info.get('usage'),
                 downloaded=pd.Timestamp.now().isoformat(timespec='seconds'))
    save_state(out_dir, state)
    return state


def ensure_job(out_dir, make_cube, title, **job_format):
    # The job's record, running the job first unless its files are already downloaded.
    state = load_state(out_dir)
    if state.get('downloaded'):
        return state
    return run_batch(get_con(), make_cube(), out_dir, title, **job_format)


# CDSE's worked example (March 2026 rates; they may change): 5,099 CPU-seconds cost 2.3 credits,
# 13,400,773 MB-seconds of memory cost 3.9 credits, and every batch job costs 2 credits up front.
CREDITS_PER_CPU_S = 2.3 / 5099
CREDITS_PER_MB_S = 3.9 / 13_400_773
BATCH_JOB_CREDITS = 2


def job_credits(state):
    # Credits a finished job used: the cost openEO reports, or an estimate from its CPU and memory use.
    if state.get('costs') is not None:
        return float(state['costs']), 'reported by openEO'
    usage = state.get('usage') or {}
    cpu, mem = usage.get('cpu') or {}, usage.get('memory') or {}
    if str(cpu.get('unit', '')).lower().startswith('cpu') and str(mem.get('unit', '')).lower().startswith('mb'):
        credits = cpu['value'] * CREDITS_PER_CPU_S + mem['value'] * CREDITS_PER_MB_S + BATCH_JOB_CREDITS
        return credits, 'estimated from CPU and memory use at March 2026 rates'
    return None, 'not reported'


# --- Reading the downloads ------------------------------------------------------------------------
def dated_tifs(folder):
    files = {}
    for path in sorted(Path(folder).glob('*.tif')):
        found = re.search(r'(\d{4}-\d{2}-\d{2})', path.name)
        if found:
            files[found.group(1)] = path
    return files


def composite_files(folder):
    # {window label: file}; each file's date is matched to the 5-day window it falls in.
    files = {}
    for date, path in dated_tifs(folder).items():
        w = (pd.Timestamp(date) - pd.Timestamp(COMPOSITE_START)).days // WINDOW_DAYS
        if 0 <= w < len(LABELS):
            files[LABELS[w]] = path
    return files


def read_raw(path, names, window=None):
    # Bands `names` as float32 (bands, rows, cols), NaN where there is no data.
    with rasterio.open(path) as src:
        data = src.read(window=window, boundless=window is not None, fill_value=src.nodata or 0).astype('float32')
        nodata, descriptions = src.nodata, [d or '' for d in src.descriptions]
        is_int = np.issubdtype(np.dtype(src.dtypes[0]), np.integer)
    if all(n in descriptions for n in names):
        data = data[[descriptions.index(n) for n in names]]
    elif data.shape[0] != len(names):
        raise RuntimeError(f'{Path(path).name} has {data.shape[0]} bands; expected {len(names)} ({names}).')
    if nodata is not None and not np.isnan(nodata):
        data[data == nodata] = np.nan
    if is_int:
        data[data == 0] = np.nan              # 0 means "no data" in 16-bit Sentinel-2 values
    else:
        data[:, np.all(data == 0, axis=0)] = np.nan
    return data


def detect_scaling(files):
    # Values are reflectance x 10000; processing-baseline-04 values also carry +1000 unless harmonised.
    # Red is the test band: green crops reflect very little red, so its low end shows whether +1000 is there.
    reds = []
    for path in list(files.values())[:6]:
        red = read_raw(path, BANDS)[0]
        reds.append(red[np.isfinite(red)])
    values = np.concatenate(reds) if reds else np.array([])
    if values.size == 0:
        raise RuntimeError('The pilot composites contain no valid pixels.')
    p2, p99 = np.percentile(values, [2, 99])
    if p99 < 2:
        return 0.0, 1.0, f'values are already reflectance (99th percentile of red {p99:.2f})'
    if p2 > 900:
        return 1000.0, 10000.0, f'red rarely drops below 900 (2nd percentile {p2:.0f}), so the +1000 offset is present'
    return 0.0, 10000.0, f'the 2nd percentile of red is {p2:.0f}, so the values carry no offset'


def to_reflectance(raw):
    return (raw - DN_OFFSET) / SCALE


def read_window(path, window=None):
    return to_reflectance(read_raw(path, BANDS, window))


def indices(refl):
    red, nir, sw1, sw2 = refl
    with np.errstate(divide='ignore', invalid='ignore'):
        ndvi = (nir - red) / (nir + red)
        nbr = (nir - sw2) / (nir + sw2)
    mirbi = 10 * sw2 - 9.8 * sw1 + 2   # Mid-Infrared Burn Index, on reflectance 0-1
    return ndvi, nbr, mirbi


def as_image(rgb):
    # Clip to 0-1 and show a pixel grey when any of its three bands is missing, so a gap in one band
    # can't turn into a false colour.
    rgb = np.clip(rgb, 0, 1)
    rgb[~np.isfinite(rgb).all(axis=-1)] = 0.85
    return rgb


def false_colour(refl):
    # SWIR 2, NIR, red as RGB: green crops look green, bare soil pink to magenta, straw pale pink or white, and
    # fresh burns very dark (black or dark maroon). Grey means no clear view.
    return as_image(np.stack([refl[3] / 0.35, refl[1] / 0.35, refl[0] / 0.20], axis=-1))


def true_colour(blue, green, red):
    # Natural colour at 3x brightness: burns look black or dark grey, bare soil brown, straw pale yellow or white.
    return as_image(np.stack([red, green, blue], axis=-1) * 3.0)


def read_patch(path):
    # One single-date file (netCDF, one per point): pixel centres x and y in metres (north first), the dates,
    # and each band as (date, y, x) with NaN where there is no data.
    with xr.open_dataset(path) as ds:
        names = DATE_BANDS + ['SCL']
        lacking = [b for b in names if b not in ds.data_vars]
        if lacking:
            raise RuntimeError(f'{Path(path).name} lacks bands {lacking}; it has {list(ds.data_vars)}.')
        dims = ds[names[0]].dims
        if not {'x', 'y'} <= set(dims) or len(dims) != 3:
            raise RuntimeError(f'{Path(path).name}: expected dimensions (time, y, x), found {dims}.')
        tdim = next(d for d in dims if d not in ('x', 'y'))
        ds = ds[names].transpose(tdim, 'y', 'x').sortby(tdim).sortby('y', ascending=False)
        patch = {'x': ds['x'].values.astype(float), 'y': ds['y'].values.astype(float),
                 'dates': pd.to_datetime(ds[tdim].values)}
        for b in names:
            values = ds[b].values.astype('float32')
            # The file's own fill value is already NaN here. 0 is a real value: on hazy days the atmospheric
            # correction can push the blue band of green fields down to 0.
            values[(values < 0) | (values >= 65535)] = np.nan
            patch[b] = values
    return patch


def match_patches(folder, pts_xy):
    # {point_id: file}. By the point ID in the file name if openEO used it, otherwise by position: the point
    # inside the file's extent that is closest to its centre.
    ids, found = set(pts_xy['point_id']), {}
    for path in sorted(Path(folder).glob('*.nc')):
        named = [pid for pid in re.findall(r'P\d{3}', path.stem) if pid in ids]
        if named:
            found[named[-1]] = path
            continue
        with xr.open_dataset(path) as ds:
            xs, ys = ds['x'].values, ds['y'].values
        inside = pts_xy[pts_xy['x'].between(xs.min(), xs.max()) & pts_xy['y'].between(ys.min(), ys.max())]
        if len(inside):
            dist = np.hypot(inside['x'] - xs.mean(), inside['y'] - ys.mean())
            found[inside['point_id'].iloc[int(np.argmin(dist.to_numpy()))]] = path
    return found


def date_strip(pid, patch, x, y, out_path, ncols=12):
    # Every date on which the point's 60 m block is clear, true colour above SWIR false colour, DATE_STRIP_M
    # across, with a square just outside the block. Returns the number of dates shown.
    half = DATE_STRIP_M / 2
    cols = (patch['x'] >= x - half) & (patch['x'] <= x + half)
    rows = (patch['y'] >= y - half) & (patch['y'] <= y + half)
    if not cols.any() or not rows.any():
        raise RuntimeError(f'{pid}: its single-date file does not cover the point.')
    block = np.ix_(np.abs(patch['y'] - y) <= 30, np.abs(patch['x'] - x) <= 30)
    shown = []
    for k in range(len(patch['dates'])):
        scl, red = patch['SCL'][k][block], patch['B04'][k][block]
        if scl.size and (np.isin(scl, SCL_KEEP) & np.isfinite(red)).mean() >= 0.5:
            shown.append(k)
    ncols = min(ncols, max(len(shown), 1))
    pairs = max(-(-len(shown) // ncols), 1)
    # Rows per pair of dates: true colour, false colour, then a gap row (none after the last pair).
    heights = ([1, 1, 0.35] * pairs)[:-1]
    fig, grid = plt.subplots(len(heights), ncols, figsize=(ncols * 1.25, pairs * 2.9 + 0.6), squeeze=False,
                             gridspec_kw={'height_ratios': heights, 'hspace': 0.08, 'wspace': 0.06})
    for ax in grid.ravel():
        ax.set_axis_off()
    axes = np.vstack([grid[3 * p: 3 * p + 2] for p in range(pairs)])   # drop the gap rows
    sub = np.ix_(rows, cols)
    xs, ys = patch['x'][cols], patch['y'][rows]
    ext = (xs.min() - 5, xs.max() + 5, ys.min() - 5, ys.max() + 5)   # pixel centres +- 5 m
    for i, k in enumerate(shown):
        pair, col = divmod(i, ncols)
        refl = {b: to_reflectance(patch[b][k][sub]) for b in DATE_BANDS}
        images = (true_colour(refl['B02'], refl['B03'], refl['B04']),
                  false_colour(np.stack([refl['B04'], refl['B08'], refl['B11'], refl['B12']])))
        for ax, img in zip((axes[2 * pair, col], axes[2 * pair + 1, col]), images):
            ax.imshow(img, extent=ext, interpolation='nearest')
            ax.add_patch(Rectangle((x - 45, y - 45), 90, 90, fill=False, edgecolor='cyan', linewidth=1.2))
            ax.set_xlim(ext[0], ext[1])
            ax.set_ylim(ext[2], ext[3])
        axes[2 * pair, col].set_title(f'{patch["dates"][k]:%d %b}', fontsize=8)
    if not shown:
        axes[0, 0].text(0.5, 0.5, 'No clear single date: use the composite strip.', ha='center', va='center')
    for pair in range(pairs):
        for r, name in ((2 * pair, 'true colour'), (2 * pair + 1, 'SWIR false colour')):
            axes[r, 0].text(-0.08, 0.5, name, rotation=90, transform=axes[r, 0].transAxes, ha='right', va='center',
                            fontsize=7)
    fig.suptitle(f'{pid}: was the field in the square burnt on any date? (only dates with a clear view are shown)\n'
                 'Burns: black or dark grey in true colour, dark maroon to black in false colour. Soil: brown / pink. '
                 'Straw: pale / white. Damp soil: dark brown / dark blue-purple.', fontsize=8)
    fig.savefig(out_path, dpi=100, bbox_inches='tight')
    plt.close(fig)
    return len(shown)


def trace_season(files, inside, rule, sample_rate=0.02, seed=0):
    # Walk the windows in time order and record, per pixel, when it was harvested and when (if) it burned.
    shape = inside.shape
    peak = np.full(shape, -np.inf, 'float32')       # highest NDVI seen in earlier windows
    last_valid = np.full(shape, -1, 'int16')        # last window with a clear view
    harvest_w = np.full(shape, -1, 'int16')
    burn_w = np.full(shape, -1, 'int16')
    prev_w = np.full(shape, -1, 'int16')            # last clear view before the burn showed
    clear_share, diag = [], []
    rng = np.random.default_rng(seed)
    for w, label in enumerate(LABELS):
        if label not in files:
            clear_share.append(0.0)
            continue
        refl = read_window(files[label])
        ndvi, nbr, mirbi = indices(refl)
        nir = refl[1]
        valid = inside & np.isfinite(ndvi) & np.isfinite(nbr) & np.isfinite(mirbi)
        clear_share.append(float(valid.sum() / max(inside.sum(), 1)))
        low = valid & (peak >= rule['green_ndvi']) & (ndvi < rule['harvest_ndvi'])
        harvest_w[low & (harvest_w < 0)] = w
        burnt = (low & (burn_w < 0) & (nbr <= rule['burn_nbr_max']) & (mirbi >= rule['burn_mirbi_min'])
                 & (nir <= rule['burn_nir_max']))
        burn_w[burnt] = w
        prev_w[burnt] = last_valid[burnt]
        rows, cols = np.nonzero(low & (rng.random(shape, dtype='float32') < sample_rate))
        diag.append(pd.DataFrame({'window': w, 'nbr': nbr[rows, cols], 'mirbi': mirbi[rows, cols],
                                  'nir': nir[rows, cols]}))
        peak[valid] = np.maximum(peak[valid], ndvi[valid])
        last_valid[valid] = w
    return {'harvest_w': harvest_w, 'burn_w': burn_w, 'prev_w': prev_w, 'clear_share': clear_share,
            'diag': (pd.concat(diag, ignore_index=True) if diag
                     else pd.DataFrame(columns=['window', 'nbr', 'mirbi', 'nir']))}


def flagged_by(det, radius_m, burn_w, prev_w, transform, shape):
    # Burnt pixels with a detection within radius_m, made between the last clear view before the burn and
    # the window in which the burn shows. That span is generous, so missed shares are, if anything, too low.
    flagged = np.zeros(shape, bool)
    for w, grp in det.groupby('window'):
        near = features.rasterize(((geom.buffer(radius_m), 1) for geom in grp.geometry), out_shape=shape,
                                  transform=transform, fill=0, dtype='uint8').astype(bool)
        flagged |= (burn_w >= 0) & near & (prev_w <= w) & (w <= burn_w)
    return flagged


RULE = {'green_ndvi': GREEN_NDVI, 'harvest_ndvi': HARVEST_NDVI, 'burn_nbr_max': BURN_NBR_MAX,
        'burn_mirbi_min': BURN_MIRBI_MIN, 'burn_nir_max': BURN_NIR_MAX}
print(f'{len(WINDOWS)} windows of {WINDOW_DAYS} days: {LABELS[0]} to {INTERVALS[-1][1]} (end exclusive).')

## 3. Study area and pilot box

The district polygon comes from the geoBoundaries file Notebook 01 downloaded. That file's Sangrur is the pre-2021 district, which includes Malerkotla. It is simplified to about 50 m, which keeps the openEO job small. The pilot box is 10 km × 10 km around a point inside the district.

In [ ]:
district = load_district()
district_geom = district.geometry.iloc[0].simplify(0.0005, preserve_topology=True)   # WGS84, about 50 m
district_utm = gpd.GeoSeries([district_geom], crs=4326).to_crs(UTM_EPSG).iloc[0]
centre = district_utm.representative_point()
half = PILOT_KM * 1000 / 2
pilot_utm = box(centre.x - half, centre.y - half, centre.x + half, centre.y + half)
pilot_geom = gpd.GeoSeries([pilot_utm], crs=UTM_EPSG).to_crs(4326).iloc[0]
d_minx, d_miny, d_maxx, d_maxy = district_utm.bounds
bbox_km2 = (d_maxx - d_minx) * (d_maxy - d_miny) / 1e6
print(f'{DISTRICT_LABEL}: {district_utm.area / 1e6:,.0f} km2; bounding box {bbox_km2:,.0f} km2; '
      f'pilot box {PILOT_KM ** 2} km2.')

fig, ax = plt.subplots(figsize=(6, 6))
gpd.GeoSeries([district_utm], crs=UTM_EPSG).plot(ax=ax, color='#f2efe6', edgecolor='#555555')
gpd.GeoSeries([pilot_utm], crs=UTM_EPSG).plot(ax=ax, facecolor='none', edgecolor='tab:red', linewidth=2)
ax.set_title(f'{DISTRICT_LABEL} and the {PILOT_KM} km pilot box')
ax.set_axis_off()
fig.savefig(FIGS / '01_study_area.png', bbox_inches='tight')
plt.show()

## 4. Connect to openEO

Only if something still needs downloading. The first time, this prints a link (and perhaps a code): open it, sign in with your Copernicus account and approve. Later runs log in by themselves.

In [ ]:
pilot_dir, scl_dir, full_dir = S2_RAW / 'pilot', S2_RAW / 'pilot_scl_check', S2_RAW / 'district'
date_dir = S2_RAW / 'label_dates'
to_download = [d for d, wanted in ((pilot_dir, True), (scl_dir, True), (full_dir, RUN_FULL),
                                   (date_dir, RUN_FULL and DATE_STRIPS))
               if wanted and not load_state(d).get('downloaded')]
if to_download:
    get_con()
    print('Connected to openEO.')
else:
    print('All Sentinel-2 downloads needed for this run are on disk; not connecting to openEO.')

## 5. Pilot run and cloud-mask check

**Composites.** The first job builds the composites for the pilot box (or reuses its download). It reports the credits used and projects the cost and download size of the whole district. Both projections scale the pilot by the district's bounding-box area. Part of a small job's cost is fixed overhead, so the cost projection probably runs high.

**Cloud-mask check.** The second job downloads single, unmasked dates from the burning peak with their scene classes. It then counts which classes the burnt-looking pixels fall in. If many fall in a class the composites mask out (cloud shadow, say), burns are being thrown away with the clouds. In that case, add the class to `SCL_KEEP`, delete the `pilot` folder under `data/raw/sentinel2/sangrur/` and run again.

In [ ]:
pilot_state = ensure_job(pilot_dir, lambda: build_cube(get_con(), pilot_geom),
                         f'{DISTRICT} pilot composites {SEASON_YEAR}')
pilot_files = composite_files(pilot_dir)
if not pilot_files:
    raise RuntimeError(f'The pilot job finished, but no composite files were found in {pilot_dir}.')
with rasterio.open(next(iter(pilot_files.values()))) as src:
    print(f'Pilot composites: {len(pilot_files)} of {len(LABELS)} windows; {src.count} bands of {src.dtypes[0]}, '
          f'{src.width} x {src.height} px, CRS {src.crs}.')

pilot_credits, how = job_credits(pilot_state)
if PILOT_COST_OVERRIDE is not None:
    pilot_credits, how = float(PILOT_COST_OVERRIDE), 'entered in Settings'
projected = None
if pilot_credits is not None:
    projected = max(pilot_credits - BATCH_JOB_CREDITS, 0) * bbox_km2 / PILOT_KM ** 2 + BATCH_JOB_CREDITS
    print(f'Pilot cost: {pilot_credits:.1f} credits ({how}). Projected whole-district cost: about {projected:,.0f} '
          f'credits (budget {FULL_BUDGET_CREDITS:,}). Rough, and probably high.')
else:
    print('openEO did not report the pilot cost. Look it up in the openEO Web Editor (batch jobs, info button) and '
          'enter it as PILOT_COST_OVERRIDE in Settings before the whole-district run.')

pilot_mb = sum(p.stat().st_size for p in pilot_files.values()) / 1e6
projected_gb = pilot_mb * bbox_km2 / PILOT_KM ** 2 / 1e3
free_gb = shutil.disk_usage(PROJECT).free / 1e9
print(f'Pilot download: {pilot_mb:,.0f} MB. Projected whole-district download: about {projected_gb:,.1f} GB '
      f'({free_gb:,.0f} GB free on this drive).')

offset, SCALE, why = detect_scaling(pilot_files)
DN_OFFSET = float(DN_OFFSET) if DN_OFFSET is not None else offset
print(f'Scaling: offset {DN_OFFSET:.0f}, scale {SCALE:.0f} ({why}).')

In [ ]:
SCL_NAMES = {0: 'no data', 1: 'saturated or defective', 2: 'dark area', 3: 'cloud shadow', 4: 'vegetation',
             5: 'bare soil', 6: 'water', 7: 'unclassified', 8: 'cloud, medium probability',
             9: 'cloud, high probability', 10: 'thin cirrus', 11: 'snow or ice'}
scl_state = ensure_job(scl_dir, lambda: build_scl_check(get_con(), pilot_geom),
                       f'{DISTRICT} pilot cloud-mask check {SEASON_YEAR}')
counts = {}
for date, path in dated_tifs(scl_dir).items():
    raw = read_raw(path, BANDS + ['SCL'])
    refl = to_reflectance(raw[:4])
    ndvi, nbr, mirbi = indices(refl)
    burnt_like = ((ndvi < HARVEST_NDVI) & (nbr <= BURN_NBR_MAX) & (mirbi >= BURN_MIRBI_MIN) & (refl[1] <= BURN_NIR_MAX)
                  & np.isfinite(raw[4]))
    for cls, n in zip(*np.unique(raw[4][burnt_like], return_counts=True)):
        counts[int(cls)] = counts.get(int(cls), 0) + int(n)
scl_check = pd.DataFrame({'scl_class': list(counts), 'burnt_like_px': list(counts.values())}).sort_values('scl_class')
scl_check['class_name'] = scl_check['scl_class'].map(SCL_NAMES)
scl_check['kept_in_composites'] = scl_check['scl_class'].isin(SCL_KEEP)
scl_check['share'] = (scl_check['burnt_like_px'] / max(scl_check['burnt_like_px'].sum(), 1)).round(4)
scl_check.to_csv(PROCESSED / f's2_scl_check_pilot_{SEASON_YEAR}.csv', index=False)
n_dates = len(dated_tifs(scl_dir))
lost = scl_check.loc[~scl_check['kept_in_composites'], 'share'].sum()
print(f'{n_dates} dates, {scl_check["burnt_like_px"].sum():,} burnt-looking pixels (single dates, the burn '
      f'thresholds in Settings; no check that the field was green before). '
      f'Share in classes the composites mask out: {lost:.0%}.')
display(scl_check)

if len(scl_check):
    fig, ax = plt.subplots(figsize=(7, 3.2))
    colours = np.where(scl_check['kept_in_composites'], '#4c78a8', '#bab0ac')
    ax.barh(scl_check['class_name'], scl_check['share'] * 100, color=colours)
    ax.set_xlabel('Share of burnt-looking pixels (%)')
    ax.set_title('Scene class of burnt-looking pixels, pilot box (blue: kept; grey: masked out)')
    fig.savefig(FIGS / '03_scl_check.png', bbox_inches='tight')
    plt.show()

In [ ]:
# What the pilot sees: false colour (SWIR 2, NIR, red) for three windows, and median indices over the season.
shown = [LABELS[i] for i in (3, 9, 13) if LABELS[i] in pilot_files]
rows = []
for label in LABELS:
    if label in pilot_files:
        ndvi, nbr, mirbi = indices(read_window(pilot_files[label]))
        ok = np.isfinite(ndvi)
        rows.append({'window': pd.Timestamp(label), 'clear_share': ok.mean(),
                     'NDVI': np.median(ndvi[ok]) if ok.any() else np.nan,
                     'NBR': np.median(nbr[ok]) if ok.any() else np.nan,
                     'MIRBI': np.median(mirbi[ok]) if ok.any() else np.nan})
pilot_series = pd.DataFrame(rows)

fig = plt.figure(figsize=(12, 7))
for i, label in enumerate(shown):
    ax = fig.add_subplot(2, 3, i + 1)
    ax.imshow(false_colour(read_window(pilot_files[label])))
    ax.set_title(f'{pd.Timestamp(label):%d %b} window')
    ax.set_axis_off()
ax = fig.add_subplot(2, 1, 2)
for name, colour in (('NDVI', 'tab:green'), ('NBR', 'tab:brown')):
    ax.plot(pilot_series['window'], pilot_series[name], marker='o', label=f'median {name}', color=colour)
ax2 = ax.twinx()
ax2.plot(pilot_series['window'], pilot_series['MIRBI'], marker='s', color='tab:red', label='median MIRBI')
ax2.grid(False)
ax.set_ylabel('NDVI, NBR')
ax2.set_ylabel('MIRBI')
ax.legend(loc='lower left')
ax2.legend(loc='lower right')
ax.set_title('Pilot box: median index per 5-day window (clear pixels only)')
fig.tight_layout()
fig.savefig(FIGS / '02_pilot_check.png', bbox_inches='tight')
plt.show()
display(pilot_series.round(3))

## 6. Whole-district run

Runs only when `RUN_FULL = True`. It stops before starting the job if the projected cost is above `FULL_BUDGET_CREDITS` or unknown, or if the drive lacks space for the download. Until then, the rest of the notebook works on the pilot box.

In [ ]:
full_files = {}
if RUN_FULL:
    full_state = load_state(full_dir)
    if not full_state.get('downloaded'):
        if projected is None:
            raise RuntimeError('The pilot cost is unknown: set PILOT_COST_OVERRIDE in Settings first.')
        if projected > FULL_BUDGET_CREDITS:
            raise RuntimeError(f'Projected cost {projected:,.0f} credits is above FULL_BUDGET_CREDITS '
                               f'({FULL_BUDGET_CREDITS:,}). Raise the budget only if your balance allows it.')
        if projected_gb * 1.2 > free_gb:
            raise RuntimeError(f'The download may need about {projected_gb:,.1f} GB, but only {free_gb:,.1f} GB '
                               'is free on this drive. Free some space and run again.')
        full_state = run_batch(get_con(), build_cube(get_con(), district_geom), full_dir,
                               title=f'{DISTRICT} composites {SEASON_YEAR}')
    full_files = composite_files(full_dir)
    if not full_files:
        raise RuntimeError(f'The district job finished, but no composite files were found in {full_dir}.')
    used, how = job_credits(full_state)
    print(f'Whole-district composites: {len(full_files)} of {len(LABELS)} windows. Job cost: '
          f'{"%.0f credits" % used if used is not None else "not reported"} ({how}).')

if full_files:
    AREA, AREA_LABEL, files, aoi_utm = SLUG, DISTRICT_LABEL, full_files, district_utm
else:
    AREA, AREA_LABEL, files, aoi_utm = 'pilot', f'Pilot box ({PILOT_KM} km) in {DISTRICT_LABEL}', pilot_files, pilot_utm
SUFFIX = f'{AREA}_{SEASON_YEAR}'
print(f'Working area: {AREA_LABEL} ({len(files)} windows on disk).')

## 7. Trace each pixel's season

For every 20 m pixel, in window order:
- **Green:** NDVI has reached `GREEN_NDVI` in an earlier window (a standing crop).
- **Harvested:** after that, NDVI drops below `HARVEST_NDVI`. The first such window is the harvest window.
- **Burnt:** in a harvested window, NBR is at most `BURN_NBR_MAX`, MIRBI is at least `BURN_MIRBI_MIN` *and* near-infrared reflectance is at most `BURN_NIR_MAX`. Fresh char is dark in the near infrared and relatively bright in short-wave infrared 2, which pushes NBR down and MIRBI up. Straw left on the field is brighter in both.
- **Why the NIR condition:** bare soil after harvest has much the same NBR and MIRBI as a burn, but it is brighter in the near infrared. Without this condition, the pilot marked 91% of harvested land burnt (decision log, 26 Oct 2026).
- **Unconfirmed:** a burn first seen in a window starting on or after `UNCONFIRMED_FROM` (19 Nov) is mapped as a class of its own and kept out of the headline figures. At Checkpoint 3, none of the labelled points the rule calls burnt that late was a confirmed burn, and VIIRS saw far fewer fires per mapped hectare than earlier: damp soil under young wheat can pass all three tests (decision log, 30 Oct 2026).

Pixels that never go from green to harvested are left out of the analysis: trees, towns, water, unsown land, and crops still standing in December.

In [ ]:
with rasterio.open(next(iter(files.values()))) as src:
    TRANSFORM, CRS, SHAPE = src.transform, src.crs, (src.height, src.width)
aoi_raster = gpd.GeoSeries([aoi_utm], crs=UTM_EPSG).to_crs(CRS).iloc[0]
inside = features.rasterize([(aoi_raster, 1)], out_shape=SHAPE, transform=TRANSFORM, fill=0,
                            dtype='uint8').astype(bool)
season = trace_season(files, inside, RULE)
harvest_w, burn_w, prev_w = season['harvest_w'], season['burn_w'], season['prev_w']
PX_HA = RESOLUTION_M ** 2 / 10_000
harvested = harvest_w >= 0
burnt = burn_w >= 0
# Burns first seen from UNCONFIRMED_FROM are unconfirmed: mapped, but kept out of the headline figures.
late_from = next((w for w, label in enumerate(LABELS) if UNCONFIRMED_FROM and label >= UNCONFIRMED_FROM),
                 len(LABELS))
late_day = f'{pd.Timestamp(LABELS[late_from]):%d %b}' if late_from < len(LABELS) else None
unconfirmed = burnt & (burn_w >= late_from)
confirmed = burnt & ~unconfirmed
print(f'Harvested during the season: {harvested.sum() * PX_HA:,.0f} ha; burnt: {confirmed.sum() * PX_HA:,.0f} ha '
      f'({confirmed.sum() / max(harvested.sum(), 1):.0%} of harvested)'
      + (f', plus {unconfirmed.sum() * PX_HA:,.0f} ha first seen from {late_day} (unconfirmed).' if late_day else '.'))

diag = season['diag']
fig, axes = plt.subplots(1, 2, figsize=(12.5, 5))
panels = (('nbr', 'mirbi', BURN_NBR_MAX, BURN_MIRBI_MIN, 'Burnt needs NBR left of the line and MIRBI above it'),
          ('nir', 'nbr', BURN_NIR_MAX, BURN_NBR_MAX, '... and NIR left of the line: char is dark, bare soil is not'))
for ax, (x, y, x_line, y_line, title) in zip(axes, panels):
    if len(diag):
        hb = ax.hexbin(diag[x], diag[y], gridsize=60, bins='log', cmap='viridis', mincnt=1)
        fig.colorbar(hb, ax=ax, label='observations (log scale)')
    ax.axvline(x_line, color='tab:red', linestyle='--')
    ax.axhline(y_line, color='tab:red', linestyle='--')
    ax.set_xlabel(x.upper())
    ax.set_ylabel(y.upper())
    ax.set_title(title, fontsize=10)
fig.suptitle('Harvested-field observations and the burn thresholds in Settings')
fig.savefig(FIGS / '07_burn_rule_diagnostic.png', bbox_inches='tight')
plt.show()

## 8. Fire detections and the missed share

This step gets FIRMS detections for the district, plus a small margin, over the same dates. It uses all three VIIRS satellites and MODIS. A burnt pixel counts as **flagged** if a detection lies within the matching distance at any time between its last clear view before the burn and the window in which the burn shows. Three matching rules are reported:
- **VIIRS, strict:** within 375 m (one VIIRS pixel).
- **VIIRS, lenient:** within 750 m, allowing for geolocation error.
- **Any satellite:** VIIRS within 750 m, or MODIS within 1 km. Official monitoring uses both sensors.

The time span is generous, and the lenient rules more so. If anything, then, the missed share is underestimated.

The shares are of confirmed burns: unconfirmed late burns (section 7) are left out. The strict share with them included is printed for comparison.

In [ ]:
d_bounds = district.total_bounds
firms_bbox = (d_bounds[0] - 0.05, d_bounds[1] - 0.05, d_bounds[2] + 0.05, d_bounds[3] + 0.05)
period_start = pd.Timestamp(COMPOSITE_START)
period_end = pd.Timestamp(COMPOSITE_END) - pd.Timedelta(1, unit='D')
availability = firms_availability()
if availability is None:
    print('Could not check which FIRMS products cover the season; trying each sensor\'s products in order.')
frames, used, gaps = [], [], []
for sensor, candidates in FIRMS_CANDIDATES.items():
    if SEASON_YEAR < FIRMS_FIRST_SEASON.get(sensor, 0):
        print(f'{sensor}: no data for the {SEASON_YEAR} season (first season {FIRMS_FIRST_SEASON[sensor]}); skipped.')
        continue
    if availability is not None:
        plan, missing = plan_firms(candidates, period_start, period_end, availability)
        for source, lo, hi in plan:
            try:
                data = download_firms(source, lo, hi, firms_bbox)
            except SourceUnavailable as err:
                print(f'  {source} refused {lo:%Y-%m-%d} to {hi:%Y-%m-%d}: {err}')
                missing.append((lo, hi))
                continue
            used.append((sensor, source, lo, hi))
            if len(data):
                frames.append(data.assign(sensor=sensor))
        gaps += [(sensor, a, b) for a, b in missing]
    else:
        for source in candidates:
            try:
                data = download_firms(source, period_start, period_end, firms_bbox)
            except SourceUnavailable as err:
                print(f'  {source}: {err}; trying the next product.')
                continue
            used.append((sensor, source, period_start, period_end))
            if len(data):
                frames.append(data.assign(sensor=sensor))
            break
        else:
            gaps.append((sensor, period_start, period_end))
for sensor, source, lo, hi in used:
    print(f'{sensor}: {source}, {lo:%d %b} to {hi:%d %b %Y}')
for sensor, a, b in gaps:
    print(f'WARNING: no FIRMS product covers {sensor} from {a:%d %b} to {b:%d %b %Y}. Missed shares will be too high.')

columns = ['latitude', 'longitude', 'acq_date', 'acq_time', 'source', 'sensor', 'type']
raw_det = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame(columns=columns)
for col in columns:
    if col not in raw_det:
        raw_det[col] = np.nan
det = raw_det.drop_duplicates(subset=['latitude', 'longitude', 'acq_date', 'acq_time', 'source'])
det = det[det['type'].isna() | (det['type'] == 0)].copy()   # keep presumed vegetation fires
det['acq_date'] = pd.to_datetime(det['acq_date'])
det['window'] = (det['acq_date'] - period_start).dt.days // WINDOW_DAYS
det = gpd.GeoDataFrame(det, geometry=gpd.points_from_xy(det['longitude'], det['latitude']), crs=4326).to_crs(CRS)
det = det[(det['window'] >= 0) & (det['window'] < len(LABELS))]
det = det[det.geometry.within(aoi_raster.buffer(1500))].copy()
det['inside'] = det.geometry.within(aoi_raster)
viirs = det[det['sensor'].str.startswith('VIIRS')]
modis = det[det['sensor'] == 'MODIS']
print(f'Detections used (area plus 1.5 km): VIIRS {len(viirs):,}, MODIS {len(modis):,}.')

RULE_NAMES = {f'viirs_{r}m': f'VIIRS within {r} m' for r in VIIRS_RADII_M}
lenient = max(VIIRS_RADII_M)
RULE_NAMES[f'viirs_{lenient}m_or_modis_{MODIS_RADIUS_M}m'] = (f'VIIRS within {lenient} m or MODIS within '
                                                             f'{MODIS_RADIUS_M / 1000:g} km')
flags = {f'viirs_{r}m': flagged_by(viirs, r, burn_w, prev_w, TRANSFORM, SHAPE) for r in VIIRS_RADII_M}
flags[f'viirs_{lenient}m_or_modis_{MODIS_RADIUS_M}m'] = flags[f'viirs_{lenient}m'] | \
    flagged_by(modis, MODIS_RADIUS_M, burn_w, prev_w, TRANSFORM, SHAPE)
STRICT = f'viirs_{min(VIIRS_RADII_M)}m'

burnt_ha = confirmed.sum() * PX_HA          # the headline counts confirmed burns only
missed = {k: (1 - (flag & confirmed).sum() / confirmed.sum()) if confirmed.any() else np.nan
          for k, flag in flags.items()}
for k, share in missed.items():
    print(f'Missed share, {RULE_NAMES[k]}: {share:.0%} of {burnt_ha:,.0f} burnt ha')
missed_all = (1 - (flags[STRICT] & burnt).sum() / burnt.sum()) if burnt.any() else np.nan
if unconfirmed.any():
    print(f'With the {unconfirmed.sum() * PX_HA:,.0f} unconfirmed ha included: {missed_all:.0%} '
          f'({RULE_NAMES[STRICT]}).')

fig, ax = plt.subplots(figsize=(7, 3.2))
names, values = [RULE_NAMES[k] for k in missed][::-1], [v * 100 for v in missed.values()][::-1]
ax.barh(names, values, color=['#bab0ac', '#f58518', '#e45756'][-len(values):])
for i, v in enumerate(values):
    ax.text(v + 1, i, f'{v:.0f}%', va='center')
ax.set_xlim(0, 105)
ax.set_xlabel('Burnt area with no matching fire detection (%)')
ax.set_title(f'{AREA_LABEL}, {SEASON_YEAR} season' + (f' (burns first seen before {late_day})' if late_day else ''))
fig.savefig(FIGS / '06_missed_share.png', bbox_inches='tight')
plt.show()

## 9. Maps, timeline and outputs

The class map uses the strict VIIRS rule and shows unconfirmed late burns as a class of their own. The headline figures and the 1 km grid count confirmed burns; the grid gives unconfirmed hectares in a column of their own. The grid feeds both dashboard tabs:
- **Straw buyer:** how much was harvested in each square, when, and how much of it burned.
- **Agriculture department:** where burning happened that the fire counts didn't flag.

In [ ]:
classes = np.zeros(SHAPE, 'uint8')
classes[inside] = 1
classes[harvested & ~burnt] = 2
classes[confirmed & flags[STRICT]] = 3
classes[confirmed & ~flags[STRICT]] = 4
classes[unconfirmed] = 5
CLASS_NAMES = {1: 'not a harvested crop', 2: 'harvested, not burnt', 3: 'burnt, flagged by VIIRS',
               4: 'burnt, missed by VIIRS', 5: f'burnt from {late_day}, unconfirmed' if late_day else 'unconfirmed'}
CLASS_COLOURS = ['#ffffff', '#e3e3e3', '#9ecae1', '#fdae61', '#b2182b', '#8073ac']   # distinct in lightness and hue

fig, ax = plt.subplots(figsize=(8, 8))
ax.imshow(classes, cmap=ListedColormap(CLASS_COLOURS), vmin=0, vmax=5, interpolation='nearest')
shown_classes = [k for k in CLASS_NAMES if k < 5 or unconfirmed.any()]
ax.legend(handles=[Patch(color=CLASS_COLOURS[k], label=CLASS_NAMES[k]) for k in shown_classes], loc='upper center',
          bbox_to_anchor=(0.5, -0.01), ncol=2, fontsize=8, frameon=False)
ax.set_title(f'{AREA_LABEL}, {SEASON_YEAR} season: burnt area from Sentinel-2\n'
             f'(flagged = VIIRS detection within {min(VIIRS_RADII_M)} m)')
ax.set_axis_off()
fig.savefig(FIGS / '04_burnt_area_map.png', bbox_inches='tight')
plt.show()

n_w = len(LABELS)
by_window = pd.DataFrame({'window_start': LABELS, 'clear_share_of_area': np.round(season['clear_share'], 3)})
by_window['harvested_ha'] = [(harvest_w == w).sum() * PX_HA for w in range(n_w)]
by_window['harvested_not_burnt_ha'] = [((harvest_w == w) & ~burnt).sum() * PX_HA for w in range(n_w)]
by_window['burnt_ha'] = [(burn_w == w).sum() * PX_HA for w in range(n_w)]
by_window['burnt_flagged_ha'] = [((burn_w == w) & flags[STRICT]).sum() * PX_HA for w in range(n_w)]
by_window['burnt_missed_ha'] = by_window['burnt_ha'] - by_window['burnt_flagged_ha']
by_window['viirs_detections_inside'] = [int(((viirs['window'] == w) & viirs['inside']).sum()) for w in range(n_w)]
by_window['modis_detections_inside'] = [int(((modis['window'] == w) & modis['inside']).sum()) for w in range(n_w)]
by_window['burns_confirmed'] = np.arange(n_w) < late_from

fig, ax = plt.subplots(figsize=(10, 4.5))
x = pd.to_datetime(by_window['window_start'])
late_rows = ~by_window['burns_confirmed']
flagged_ha = by_window['burnt_flagged_ha'].where(~late_rows, 0)
ax.bar(x, flagged_ha, width=4, color='#fdae61', label='burnt, flagged by VIIRS')
ax.bar(x, by_window['burnt_missed_ha'].where(~late_rows, 0), width=4, bottom=flagged_ha, color='#b2182b',
       label='burnt, missed by VIIRS')
if late_rows.any():
    ax.bar(x[late_rows], by_window.loc[late_rows, 'burnt_ha'], width=4, color='#8073ac', label=CLASS_NAMES[5])
ax.plot(x, by_window['harvested_ha'], color='#3182bd', marker='o', label='harvested (first seen)')
ax.set_ylabel('Hectares per 5-day window')
ax2 = ax.twinx()
ax2.plot(x, by_window['viirs_detections_inside'], color='black', linestyle='--', marker='x', label='VIIRS detections')
ax2.set_ylabel('VIIRS detections')
ax2.grid(False)
ax.legend(loc='upper left', fontsize=8)
ax2.legend(loc='upper right', fontsize=8)
ax.set_title('When fields were harvested and burned, and what VIIRS saw')
fig.savefig(FIGS / '05_timeline.png', bbox_inches='tight')
plt.show()

with rasterio.open(PROCESSED / f's2_burn_classes_{SUFFIX}.tif', 'w', driver='GTiff', height=SHAPE[0],
                   width=SHAPE[1], count=4, dtype='uint8', crs=CRS, transform=TRANSFORM, compress='deflate') as dst:
    dst.write(classes, 1)
    dst.write((burn_w + 1).astype('uint8'), 2)       # 0 = never burnt; n = window n (1 = first window)
    dst.write((harvest_w + 1).astype('uint8'), 3)    # 0 = never harvested
    dst.write((prev_w + 1).astype('uint8'), 4)       # last clear view before the burn; 0 = not burnt
    dst.descriptions = ('class', 'burn_window', 'harvest_window', 'last_clear_before_burn')
by_window.to_csv(PROCESSED / f's2_burn_by_window_{SUFFIX}.csv', index=False)

gap_days = (burn_w[confirmed] - harvest_w[confirmed]) * WINDOW_DAYS
pilot_used, _ = job_credits(pilot_state)
full_used = job_credits(load_state(full_dir))[0] if full_files else None
summary = {
    'area': AREA_LABEL,
    'area_km2': round(inside.sum() * RESOLUTION_M ** 2 / 1e6, 1),
    'season_windows': f'{LABELS[0]} to {INTERVALS[-1][1]} (end exclusive), {WINDOW_DAYS}-day median composites',
    'mean_clear_share': round(float(np.mean(season['clear_share'])), 3),
    'harvested_ha': round(harvested.sum() * PX_HA, 1),
    'burnt_ha': round(burnt_ha, 1),
    'burnt_unconfirmed_ha': round(unconfirmed.sum() * PX_HA, 1),
    'harvested_not_burnt_ha': round((harvested & ~burnt).sum() * PX_HA, 1),
    'burnt_share_of_harvested': round(confirmed.sum() / max(harvested.sum(), 1), 4),
    'median_days_harvest_to_burn': float(np.median(gap_days)) if gap_days.size else np.nan,
    **{f'missed_share_{k}': round(v, 4) for k, v in missed.items()},
    f'missed_share_{STRICT}_including_unconfirmed': round(missed_all, 4),
    'viirs_detections_inside': int(viirs['inside'].sum()),
    'modis_detections_inside': int(modis['inside'].sum()),
    'firms_products': '; '.join(f'{s}: {src} {lo:%Y-%m-%d} to {hi:%Y-%m-%d}' for s, src, lo, hi in used),
    'firms_gaps': '; '.join(f'{s}: {a:%Y-%m-%d} to {b:%Y-%m-%d}' for s, a, b in gaps) or 'none',
    'rule': json.dumps(RULE),
    'unconfirmed_from': UNCONFIRMED_FROM or 'none',
    'rule_status': (RULE_STATUS if SEASON_YEAR == VALIDATED_SEASON else
                    f'{RULE_STATUS} on the {VALIDATED_SEASON} labels; applied unchanged to {SEASON_YEAR}, '
                    'which has no labels') + (f'; burns first seen from {late_day} unconfirmed' if late_day else ''),
    'scl_classes_kept': ','.join(map(str, SCL_KEEP)),
    'scaling': f'offset {DN_OFFSET:.0f}, scale {SCALE:.0f}',
    'openeo_credits_pilot': round(pilot_used, 1) if pilot_used is not None else 'not reported',
    'openeo_credits_district': round(full_used, 1) if full_used is not None else 'not run or not reported',
}
summary_table = pd.DataFrame({'metric': list(summary), 'value': list(summary.values())})
summary_table.to_csv(PROCESSED / f's2_burn_summary_{SUFFIX}.csv', index=False)

# 1 km grid for the dashboard: 50 x 50 pixels per square, aligned with the composites.
block = int(1000 / RESOLUTION_M)
pad = ((0, (-SHAPE[0]) % block), (0, (-SHAPE[1]) % block))

def block_sum(values):
    padded = np.pad(values.astype('float64'), pad)
    return padded.reshape(padded.shape[0] // block, block, padded.shape[1] // block, block).sum(axis=(1, 3))

sums = {'inside_px': block_sum(inside), 'harvested_px': block_sum(harvested), 'burnt_px': block_sum(confirmed),
        'unconfirmed_px': block_sum(unconfirmed), 'harvested_not_burnt_px': block_sum(harvested & ~burnt),
        'missed_px': block_sum(confirmed & ~flags[STRICT]),
        'harvest_w_sum': block_sum(np.where(harvested, harvest_w, 0)),
        'burn_w_sum': block_sum(np.where(confirmed, burn_w, 0))}

def window_date(mean_w):
    return WINDOWS[int(round(mean_w))].strftime('%Y-%m-%d') if np.isfinite(mean_w) else None

cells = []
for (r, c), n_inside in np.ndenumerate(sums['inside_px']):
    if n_inside == 0:
        continue
    x0, y0 = TRANSFORM * (c * block, r * block)
    x1, y1 = TRANSFORM * ((c + 1) * block, (r + 1) * block)
    n_harv, n_burnt = sums['harvested_px'][r, c], sums['burnt_px'][r, c]
    cells.append({
        'cell': f'r{r:03d}c{c:03d}', 'share_in_area': round(n_inside / block ** 2, 3),
        'harvested_ha': round(n_harv * PX_HA, 2), 'burnt_ha': round(n_burnt * PX_HA, 2),
        'burnt_unconfirmed_ha': round(sums['unconfirmed_px'][r, c] * PX_HA, 2),
        'harvested_not_burnt_ha': round(sums['harvested_not_burnt_px'][r, c] * PX_HA, 2),
        'burnt_missed_ha': round(sums['missed_px'][r, c] * PX_HA, 2),
        'mean_harvest_window_start': window_date(sums['harvest_w_sum'][r, c] / n_harv if n_harv else np.nan),
        'mean_burn_window_start': window_date(sums['burn_w_sum'][r, c] / n_burnt if n_burnt else np.nan),
        'geometry': box(min(x0, x1), min(y0, y1), max(x0, x1), max(y0, y1))})
grid = gpd.GeoDataFrame(cells, geometry='geometry', crs=CRS).to_crs(4326)
grid.to_file(PROCESSED / f's2_burn_grid_1km_{SUFFIX}.geojson', driver='GeoJSON',
             layer_options={'COORDINATE_PRECISION': 5})
det_out = det.to_crs(4326)
det_out = det_out.assign(longitude=det_out.geometry.x.round(5), latitude=det_out.geometry.y.round(5),
                         acq_date=det_out['acq_date'].dt.strftime('%Y-%m-%d'))
det_out[['latitude', 'longitude', 'acq_date', 'acq_time', 'sensor', 'source', 'window', 'inside']].to_csv(
    PROCESSED / f'firms_detections_{SUFFIX}.csv', index=False)
print(f'Wrote outputs for "{AREA}": classes GeoTIFF, summary, by-window table, {len(grid):,} grid squares, '
      f'{len(det_out):,} detections.')
display(summary_table)

In [ ]:
register_source(
    source_id='sentinel2_l2a_cdse_openeo' + ('' if SEASON_YEAR == VALIDATED_SEASON else f'_{SEASON_YEAR}'),
    dataset_or_product='Sentinel-2 L2A surface reflectance (B04, B08, B11, B12) and scene classification (SCL)',
    provider='Copernicus Data Space Ecosystem (openEO), European Union / ESA',
    url='https://dataspace.copernicus.eu/', date_accessed=ACCESS_DATE,
    geography=f'{DISTRICT_LABEL}, {STATE}', time_range=f'{COMPOSITE_START} to {COMPOSITE_END} (exclusive)',
    resolution=f'{RESOLUTION_M} m (nearest neighbour from 10 m and 20 m bands); {WINDOW_DAYS}-day median composites',
    licence_or_terms=('Free and open Copernicus data licence; credit "Contains modified Copernicus Sentinel data '
                      f'{SEASON_YEAR}"'),
    preprocessing=(f'Scene classes kept: {", ".join(map(str, SCL_KEEP))}; scenes over {MAX_CLOUD_COVER}% cloud '
                   'skipped; median per window; clipped to the district polygon; reflectance offset checked on the '
                   'pilot box'),
    known_limitations='Cloud and smoke can hide burns; scars can be ploughed under within days; the scene '
                      'classification can mislabel burnt fields; damp soil under a young crop can pass the '
                      'burn tests, so burns first seen late in the season are reported as unconfirmed',
    used_in=NOTEBOOK)
for sensor, source, lo, hi in used:
    is_modis = sensor == 'MODIS'
    register_source(
        source_id=f'firms_{source.lower()}_{SLUG}_{SEASON_YEAR}',
        dataset_or_product=(f'{"MODIS 1 km" if is_modis else "VIIRS 375 m"} active fire detections; '
                            f'FIRMS API source {source}'),
        provider='NASA FIRMS (Fire Information for Resource Management System)',
        url='https://firms.modaps.eosdis.nasa.gov/api/area/', date_accessed=ACCESS_DATE,
        geography=f'Bounding box of {DISTRICT_LABEL} plus 0.05 degrees',
        time_range=f'{lo:%Y-%m-%d} to {hi:%Y-%m-%d}',
        resolution='1 km fire pixels' if is_modis else '375 m fire pixels',
        licence_or_terms='NASA open data policy; cite per the FIRMS citation policy',
        preprocessing='Duplicates removed; vegetation-fire type kept where the field exists; assigned to 5-day windows',
        known_limitations=('Detections only at satellite overpass times; fires under cloud or smoke, or too small, '
                           'are missed'),
        used_in=NOTEBOOK)
print('Source register updated.')

## 10. Validation points and image strips (Checkpoint 3)

Runs only for the whole district, once: later runs reuse the same points and strips. Points are drawn at random from each mapped class:
- 100 from "burnt"
- 100 from "harvested, not burnt"
- 40 from everything else inside the district, to catch burns the rule missed entirely

Each point sits at the centre of a 3 × 3 block of one class (60 m across), so it isn't on a field edge. The label sheet doesn't show the map's class, so labelling stays blind.

**Two image strips per point**, in `data/raw/validation_chips/`:
- **`P001_dates.png`: single dates. Label from this one.** Every date from 20 Sep to 4 Dec on which the point had a clear view, at 10 m and 640 m across. Each pair of rows shows the same dates twice: true colour on top, short-wave infrared false colour below. The cyan square sits just outside the 60 m block you are judging, so the block itself stays visible. A last, small openEO job fetches these images (once).
- **`P001.png`: 5-day composites.** All 18 windows from 5 Sep, at 20 m, with a cross on the point. Use it for context, such as whether the field was a green crop before 20 Sep.

The single dates are the better evidence. A 5-day median can blur a burn that stayed dark for only a day or two, and the rule itself reads those medians: labels taken from them would partly repeat the rule.

**What to look for** in the single dates. Find the first clear date after the straw goes (straw or stubble is pale yellow in true colour, white or pale pink in false colour) and compare the square with the tilled fields around it on that same date:
- **`yes`, a burn:** black or near-black, filling the field up to its edges: black to dark maroon in false colour, charcoal grey in true colour. Scars fade to the colour of tilled soil within days, so check every date; the black can show on a single panel. Write the date of the first dark panel in `burn_window_seen` (optional).
- **`no`, not a burn:** the same magenta, pink or lilac as the tilled fields around it. Magenta on its own is never evidence of a burn: every harvested field ends up that colour. Damp soil after watering is navy or blue-grey in false colour (dark brown in true colour) and dries back to pink within days; a new crop shows green. A missed cloud shadow is dark too, but it covers several fields at once and lasts one date.
- **`unsure`, with a note saying why:**
  - `dull`: darker than the tilled fields around it, but not black. A burn already tilled in and freshly worked soil look the same.
  - `gap`: no clear view for a week or more between the last straw view and plain soil (most often 23 or 25 Oct to 2 Nov), so a burn could have come and gone unseen. If the field is black on the first clear date after the gap, it is still `yes`.
  - `edge`: the edge of a burn runs through the centre of the square.
  - `hazy`: the only dark view is on a hazy date.
- **Partly burnt:** `yes` when most of the square, centre included, looks burnt on some date. A burnt field beside the square doesn't count.

**Your task:** open `data/processed/validation_labels_sangrur_2025.csv`. For each point, fill in `burnt` with `yes`, `no` or `unsure`: was the field in the square burnt at any time? Use `notes` for the `unsure` reason and anything odd. Close the file in Excel before running the notebook again.

In [ ]:
points_path = PROCESSED / f'validation_points_{SUFFIX}.csv'
labels_path = PROCESSED / f'validation_labels_{SUFFIX}.csv'
if SEASON_YEAR != VALIDATED_SEASON:
    print(f'Validation points and labels belong to the {VALIDATED_SEASON} season; none are drawn for {SEASON_YEAR}.')
elif AREA == 'pilot':
    print('Validation points are drawn for the whole district only (RUN_FULL = True).')
else:
    if points_path.exists():
        pts = pd.read_csv(points_path)
        print(f'Reusing the {len(pts)} validation points in {points_path.name}.')
    else:
        from scipy import ndimage
        rng = np.random.default_rng(SAMPLE_SEED)
        # Classes 3 to 5: every burn the rule finds, unconfirmed ones included, is one stratum.
        members = {'burnt': classes >= 3, 'harvested, not burnt': classes == 2, 'other': classes == 1}
        picked = []
        for stratum, n in SAMPLE_SIZES.items():
            pure = ndimage.minimum_filter(members[stratum].astype('uint8'), size=3).astype(bool)
            rows, cols = np.nonzero(pure)
            for i in rng.choice(len(rows), size=min(n, len(rows)), replace=False):
                picked.append((stratum, int(rows[i]), int(cols[i]), int(members[stratum].sum())))
        pts = pd.DataFrame([picked[i] for i in rng.permutation(len(picked))],
                           columns=['stratum', 'row', 'col', 'stratum_pixels'])
        pts.insert(0, 'point_id', [f'P{i + 1:03d}' for i in range(len(pts))])
        pts['half'] = np.where(np.arange(len(pts)) % 2 == 0, 'A', 'B')   # A picks thresholds, B checks them
        xs, ys = rasterio.transform.xy(TRANSFORM, pts['row'], pts['col'])
        lonlat = gpd.GeoSeries(gpd.points_from_xy(xs, ys), crs=CRS).to_crs(4326)
        pts['lon'], pts['lat'] = lonlat.x.round(6), lonlat.y.round(6)
        pts['rule_at_draw'] = json.dumps(RULE)
        pts.to_csv(points_path, index=False)
        print(f'Drew {len(pts)} points: ' + ', '.join(f'{k} {v}' for k, v in pts['stratum'].value_counts().items()))

    # Composite strips: every window, false colour, 1.5 km around each point (made only where missing).
    CHIPS.mkdir(parents=True, exist_ok=True)
    todo = pts[[not (CHIPS / f'{pid}.png').exists() for pid in pts['point_id']]]
    half_px = CHIP_PX // 2
    for pid, r, c in todo[['point_id', 'row', 'col']].itertuples(index=False):
        fig, axes = plt.subplots(2, 9, figsize=(9 * 1.6, 2 * 1.85))
        for ax, label in zip(axes.ravel(), LABELS):
            if label in files:
                tile = false_colour(read_window(files[label], Window(c - half_px, r - half_px, CHIP_PX, CHIP_PX)))
            else:
                tile = np.full((CHIP_PX, CHIP_PX, 3), 0.85)
            ax.imshow(tile, interpolation='nearest')
            ax.plot([half_px], [half_px], marker='+', color='cyan', markersize=10, mew=1.5)
            ax.set_title(f'{pd.Timestamp(label):%d %b}', fontsize=8)
            ax.set_axis_off()
        fig.suptitle(f'{pid}: was the field at the cross burnt in any panel? (grey = no clear view)', fontsize=9)
        fig.savefig(CHIPS / f'{pid}.png', dpi=100, bbox_inches='tight')
        plt.close(fig)
    if len(todo):
        print(f'Made {len(todo)} composite strips in {CHIPS}.')

    # Single-date strips at 10 m, the main evidence for labelling: one small openEO job for all points, then a
    # strip per point (made only where missing).
    if DATE_STRIPS:
        pts_xy = pts[['point_id']].copy()
        pts_xy['x'], pts_xy['y'] = rasterio.transform.xy(TRANSFORM, pts['row'], pts['col'])
        date_state = ensure_job(date_dir, lambda: build_date_cube(get_con(), point_squares(pts_xy)),
                                f'{DISTRICT} label dates {SEASON_YEAR}', out_format='netCDF',
                                sample_by_feature=True, feature_id_property='point_id')
        patches = match_patches(date_dir, pts_xy)
        version_file = CHIPS / 'date_strips_version.txt'
        drawn_by = version_file.read_text().strip() if version_file.exists() else '1'   # version 1 left no file
        redraw = drawn_by != DATE_STRIP_VERSION
        if redraw and any(CHIPS.glob('P*_dates.png')):
            print(f'The single-date strips were drawn by an older version of this notebook; redrawing them all '
                  f'(version {DATE_STRIP_VERSION}). Labels are not affected.')
        shown = {}
        for pid, x, y in pts_xy[['point_id', 'x', 'y']].itertuples(index=False):
            out = CHIPS / f'{pid}_dates.png'
            if pid in patches and (redraw or not out.exists()):
                shown[pid] = date_strip(pid, read_patch(patches[pid]), x, y, out)
                if len(shown) % 40 == 0:
                    print(f'  {len(shown)} single-date strips made so far ({pd.Timestamp.now():%H:%M})')
        if patches:
            version_file.write_text(DATE_STRIP_VERSION)
        used, how = job_credits(date_state)
        print(f'Single dates {DATE_STRIP_START} to {DATE_STRIP_END}: files for {len(patches)} of {len(pts)} points; '
              f'job cost {"%.0f credits" % used if used is not None else "not reported"} ({how}).')
        if shown:
            n = np.array(list(shown.values()))
            print(f'Made {len(shown)} single-date strips; clear dates per point: median {np.median(n):.0f} '
                  f'(range {n.min()}-{n.max()}).')
        lacking = sorted(set(pts['point_id']) - set(patches))
        if lacking:
            print(f'No single-date file for {len(lacking)} points ({", ".join(lacking[:10])}'
                  f'{", ..." if len(lacking) > 10 else ""}): label those from the composite strips.')
        register_source(
            source_id='sentinel2_l2a_cdse_openeo_label_dates',
            dataset_or_product=('Sentinel-2 L2A single dates (B02, B03, B04, B08, B11, B12) and scene '
                                'classification (SCL), around the validation points'),
            provider='Copernicus Data Space Ecosystem (openEO), European Union / ESA',
            url='https://dataspace.copernicus.eu/', date_accessed=ACCESS_DATE,
            geography=f'{len(pts)} squares of {DATE_STRIP_M} m around validation points in {DISTRICT_LABEL}',
            time_range=f'{DATE_STRIP_START} to {DATE_STRIP_END} (exclusive)',
            resolution='10 m (20 m bands by nearest neighbour); single dates, no compositing',
            licence_or_terms=('Free and open Copernicus data licence; credit "Contains modified Copernicus Sentinel '
                              f'data {SEASON_YEAR}"'),
            preprocessing=(f'Scenes over {DATE_STRIP_MAX_CLOUD}% cloud skipped; a date is shown only where the scene '
                           "classification marks most of the point's 60 m block clear"),
            known_limitations=('Used only to interpret labels by eye; damp soil and missed cloud shadows can look '
                               'dark like a burn'),
            used_in=NOTEBOOK)

    if not labels_path.exists():
        sheet = pts[['point_id']].copy()
        sheet['chip_file'] = 'data/raw/validation_chips/' + sheet['point_id'] + '.png'
        sheet['lat'], sheet['lon'] = pts['lat'], pts['lon']
        sheet['burnt'], sheet['burn_window_seen'], sheet['notes'] = '', '', ''
        sheet.to_csv(labels_path, index=False)
        print(f'Label sheet ready: {labels_path}')
    sheet = pd.read_csv(labels_path, dtype=str).fillna('')
    if DATE_STRIPS and 'dates_file' not in sheet.columns:
        # Sheets made before the single-date strips existed get a column pointing to them; labels are untouched.
        sheet.insert(2, 'dates_file', 'data/raw/validation_chips/' + sheet['point_id'] + '_dates.png')
        try:
            sheet.to_csv(labels_path, index=False)
            print(f'Added the single-date strip of each point to {labels_path.name} (column dates_file).')
        except PermissionError:
            print(f'{labels_path.name} is open in another program; close it and run this cell again.')

## 11. Check the burn rule against the labels (Checkpoint 3)

Runs once at least `MIN_LABELS` points are labelled `yes` or `no`. The points were split in two halves when they were drawn:
- **Half A** fits thresholds. Every combination of NBR, MIRBI and NIR thresholds is applied to the points' own time series; the one with the best F1 score is the *suggested* rule.
- **Half B** checks rules on points they weren't fitted to. Accuracy is weighted by the area each sampling class covers, using the stratified estimator of Stehman (2014). That is the general form of the Olofsson et al. (2014) approach, for when the map being scored isn't the one the sample was drawn from.

**Checkpoint 3 (30 Oct 2026):** the suggested rule did worse on half B than the thresholds already in Settings, so those stay (decision log). No labels went into choosing them, so the rule in Settings is also scored on all labelled points, which gives tighter figures. Thresholds fitted on half A are only ever judged on half B.

What it writes:
- **Accuracy** (`validation_accuracy_…csv`): the suggested rule, the thresholds in Settings counting every burn, and the **map as reported**, in which burns first seen from `UNCONFIRMED_FROM` count as not burnt.
- **Burnt area from the labels alone** (`validation_area_estimate_…csv`), with a 95% confidence interval: an independent check on the map's figure.
- **Unsure points, both ways** (`validation_unsure_bounds_…csv`). Leaving them out can flatter the accuracy, because the hard cases go. So the map as reported is scored again with every `unsure` counted as not burnt, then as burnt, and the area file gives the burnt area from the labels for the same three cases. The notes give the reasons (`gap`, `dull`, `edge`, `hazy`).
- **Late windows** (`validation_late_window_check_…csv`): the window in which the rule first calls each labelled point burnt, before or from `UNCONFIRMED_FROM`, against its label. This check is why burns from 19 Nov are unconfirmed.

Nothing here changes the map. To change the thresholds, edit them in Settings and run again.

In [ ]:
labels = pd.DataFrame()
if AREA != 'pilot' and SEASON_YEAR == VALIDATED_SEASON and labels_path.exists():
    labels = pd.read_csv(labels_path, dtype=str).fillna('')
    if 'notes' not in labels.columns:
        labels['notes'] = ''
answers = labels['burnt'].str.strip().str.lower() if len(labels) else pd.Series(dtype=str)
usable = labels[answers.isin(['yes', 'no'])] if len(labels) else labels
if SEASON_YEAR != VALIDATED_SEASON:
    print(f'The rule is checked on the {VALIDATED_SEASON} labels only; nothing to check for {SEASON_YEAR}.')
elif AREA == 'pilot' or len(usable) < MIN_LABELS:
    print(f'The rule check waits for at least {MIN_LABELS} labelled points (now {len(usable)}).')
else:
    # Every answered point is read. 'unsure' points stay out of the fitting and the main figures, and come back
    # for the bounds at the end.
    answered = labels[answers.isin(['yes', 'no', 'unsure'])]
    pts = pd.read_csv(points_path).merge(answered[['point_id', 'burnt', 'notes']], on='point_id')
    answer = pts['burnt'].str.strip().str.lower().to_numpy()
    sure = answer != 'unsure'
    truth = answer == 'yes'
    xs, ys = rasterio.transform.xy(TRANSFORM, pts['row'], pts['col'])
    series = {k: np.full((len(LABELS), len(pts)), np.nan, 'float32') for k in ('ndvi', 'nbr', 'mirbi', 'nir')}
    for w, label in enumerate(LABELS):
        if label in files:
            with rasterio.open(files[label]) as src:
                raw = np.array(list(src.sample(zip(xs, ys))), dtype='float32').T
                names = [d or '' for d in src.descriptions]
                if all(b in names for b in BANDS):
                    raw = raw[[names.index(b) for b in BANDS]]
                bad = (raw == 0) if np.issubdtype(np.dtype(src.dtypes[0]), np.integer) else np.zeros(raw.shape, bool)
                if src.nodata is not None and not np.isnan(src.nodata):
                    bad |= raw == src.nodata
            raw[bad] = np.nan
            refl = to_reflectance(raw)
            for k, v in zip(('ndvi', 'nbr', 'mirbi', 'nir'), (*indices(refl), refl[1])):
                series[k][w] = v
    valid = np.isfinite(series['ndvi']) & np.isfinite(series['nbr']) & np.isfinite(series['mirbi'])
    peak_before = np.vstack([np.full((1, len(pts)), -np.inf),
                             np.fmax.accumulate(np.where(valid, series['ndvi'], -np.inf), axis=0)[:-1]])
    eligible = valid & (peak_before >= GREEN_NDVI) & (series['ndvi'] < HARVEST_NDVI)

    def burn_hits(t_nbr, t_mirbi, t_nir):
        return eligible & (series['nbr'] <= t_nbr) & (series['mirbi'] >= t_mirbi) & (series['nir'] <= t_nir)

    def predict(t_nbr, t_mirbi, t_nir):
        return burn_hits(t_nbr, t_mirbi, t_nir).any(axis=0)

    def scores(pred, sel, t=truth):
        tp, fp, fn = (pred & t & sel).sum(), (pred & ~t & sel).sum(), (~pred & t & sel).sum()
        precision = tp / (tp + fp) if tp + fp else 0.0
        recall = tp / (tp + fn) if tp + fn else 0.0
        return precision, recall, (2 * precision * recall / (precision + recall) if precision + recall else 0.0)

    def weighted_accuracy(pred, sel, t=truth):
        # Area-weighted user's, producer's and overall accuracy for 'burnt' (Stehman 2014 stratified estimator).
        weights = pts.groupby('stratum')['stratum_pixels'].first()
        weights = weights / weights.sum()
        cm, var_oa = np.zeros((2, 2)), 0.0
        for stratum, w_h in weights.items():
            in_h = (pts['stratum'] == stratum).to_numpy() & sel
            n_h = in_h.sum()
            if n_h == 0:
                continue
            for i, p in enumerate((True, False)):
                for j, tv in enumerate((True, False)):
                    cm[i, j] += w_h * ((pred == p) & (t == tv) & in_h).sum() / n_h
            correct = ((pred == t) & in_h).sum() / n_h
            var_oa += w_h ** 2 * correct * (1 - correct) / max(n_h - 1, 1)
        ua = cm[0, 0] / cm[0].sum() if cm[0].sum() else np.nan
        pa = cm[0, 0] / cm[:, 0].sum() if cm[:, 0].sum() else np.nan
        return ua, pa, cm[0, 0] + cm[1, 1], np.sqrt(var_oa)

    def label_area(t, sel):
        # Burnt area from the labels alone: stratified estimate with a 95% confidence interval, in hectares.
        est, var = 0.0, 0.0
        for stratum in pts['stratum'].unique():
            in_h = (pts['stratum'] == stratum).to_numpy() & sel
            n_h = in_h.sum()
            if n_h == 0:
                continue
            N_h = pts.loc[pts['stratum'] == stratum, 'stratum_pixels'].iloc[0]
            p_h = t[in_h].mean()
            est += N_h * p_h
            var += N_h ** 2 * p_h * (1 - p_h) / max(n_h - 1, 1)
        half_width = 1.96 * np.sqrt(var)
        return est * PX_HA, (est - half_width) * PX_HA, (est + half_width) * PX_HA

    def score(pred, sel, t=truth):
        p, r, f1 = scores(pred, sel, t)
        ua, pa, oa, oa_se = weighted_accuracy(pred, sel, t)
        return {'points': int(sel.sum()), 'precision': p, 'recall': r, 'f1': f1, 'users_accuracy_burnt': ua,
                'producers_accuracy_burnt': pa, 'overall_accuracy': oa, 'overall_accuracy_se': oa_se}

    in_b = (pts['half'] == 'B').to_numpy()
    every = np.ones(len(pts), bool)
    half_a = (pts['half'] == 'A').to_numpy() & sure
    half_b = in_b & sure
    results = []
    for t_nbr in np.round(np.arange(-0.30, 0.101, 0.025), 3):
        for t_mirbi in np.round(np.arange(1.30, 2.301, 0.05), 2):
            for t_nir in np.round(np.arange(0.08, 0.201, 0.01), 2):
                results.append((t_nbr, t_mirbi, t_nir, *scores(predict(t_nbr, t_mirbi, t_nir), half_a)))
    grid_scores = pd.DataFrame(results, columns=['burn_nbr_max', 'burn_mirbi_min', 'burn_nir_max', 'precision_A',
                                                 'recall_A', 'f1_A'])
    best = grid_scores.sort_values(['f1_A', 'precision_A'], ascending=False).iloc[0]

    # The thresholds in Settings, and the map as reported: burns first seen from UNCONFIRMED_FROM count as not burnt.
    in_settings = (BURN_NBR_MAX, BURN_MIRBI_MIN, BURN_NIR_MAX)
    suggested = (best['burn_nbr_max'], best['burn_mirbi_min'], best['burn_nir_max'])
    fitted_on_a = bool(np.allclose(in_settings, suggested))
    hits = burn_hits(*in_settings)
    first_hit = np.where(hits.any(axis=0), hits.argmax(axis=0), -1)
    rule_burnt = first_hit >= 0
    reported = rule_burnt & (first_hit < late_from)
    reported_name = f'map as reported (burns from {late_day} unconfirmed)' if late_day else 'map as reported'
    scored_on = [('half B', in_b)] + ([] if fitted_on_a else [('all labelled points', every)])
    threshold_cols = ['burn_nbr_max', 'burn_mirbi_min', 'burn_nir_max']
    rows = [{'rule': 'suggested rule (best F1 on half A)', 'scored_on': 'half B',
             **dict(zip(threshold_cols, suggested)), **score(predict(*suggested), half_b)}]
    variants = [('thresholds in Settings, every burn', rule_burnt)] if late_day else []
    for name, pred in variants + [(reported_name, reported)]:
        for where, sel in scored_on:
            rows.append({'rule': name, 'scored_on': where, **dict(zip(threshold_cols, in_settings)),
                         **score(pred, sel & sure)})
    accuracy = pd.DataFrame(rows)

    # Unsure points both ways: the map as reported, and the burnt area from the labels alone (95% CI).
    cases = (('unsure left out', truth, sure), ('unsure counted as not burnt', truth, every),
             ('unsure counted as burnt', truth | ~sure, every))
    bounds = pd.DataFrame([{'scored_on': where, 'case': case, **score(reported, sel & keep, t)}
                           for where, sel in scored_on for case, t, keep in cases])
    area_est = pd.DataFrame([{'case': case, 'points': int(keep.sum()), 'unsure_points': int((~sure).sum()),
                              **dict(zip(('burnt_ha_from_labels', 'ci95_low_ha', 'ci95_high_ha'),
                                         label_area(t, keep)))} for case, t, keep in cases])
    area_est['burnt_ha_from_map'] = burnt_ha
    area_est['burnt_ha_from_map_including_unconfirmed'] = burnt.sum() * PX_HA
    note = pts['notes'].fillna('').str.lower()
    reason = np.select([note.str.contains('gap'), note.str.contains('dull'), note.str.contains('edge'),
                        note.str.contains('haz')], ['gap', 'dull', 'edge', 'hazy'], 'other')
    reasons = pd.Series(reason[~sure]).value_counts()

    # Late windows: where the rule in Settings first calls each labelled point burnt, against the label.
    early_name, late_name = (f'burnt before {late_day}', f'burnt from {late_day}') if late_day else ('burnt', 'burnt')
    when = np.where(first_hit < 0, 'not burnt', np.where(first_hit >= late_from, late_name, early_name))
    late_check = (pd.crosstab(pd.Series(when, name='rule in Settings'), pd.Series(answer, name='label'))
                  .reindex(columns=['yes', 'no', 'unsure'], fill_value=0))

    grid_scores.to_csv(PROCESSED / f'burn_rule_calibration_grid_{SUFFIX}.csv', index=False)
    accuracy.to_csv(PROCESSED / f'validation_accuracy_{SUFFIX}.csv', index=False)
    area_est.to_csv(PROCESSED / f'validation_area_estimate_{SUFFIX}.csv', index=False)
    bounds.to_csv(PROCESSED / f'validation_unsure_bounds_{SUFFIX}.csv', index=False)
    late_check.to_csv(PROCESSED / f'validation_late_window_check_{SUFFIX}.csv')
    print(f'{sure.sum()} points labelled yes or no ({truth.sum()} burnt) and {(~sure).sum()} unsure; '
          f'half A {half_a.sum()}, half B {half_b.sum()}.')
    print(f'Suggested rule (best F1 on half A): NBR <= {suggested[0]:.3f}, MIRBI >= {suggested[1]:.2f} and '
          f'NIR <= {suggested[2]:.2f}' + (', the thresholds in Settings.' if fitted_on_a else '.'))
    print('Accuracy, with unsure points left out:')
    display(accuracy.round(3))
    head = area_est.iloc[0]
    print(f'Burnt area from the labels: {head["burnt_ha_from_labels"]:,.0f} ha (95% CI {head["ci95_low_ha"]:,.0f} '
          f'to {head["ci95_high_ha"]:,.0f}); the map says {burnt_ha:,.0f} ha'
          + (f', or {burnt.sum() * PX_HA:,.0f} ha with the unconfirmed late burns.' if unconfirmed.any() else '.'))
    print('\nUnsure points by reason: ' + (', '.join(f'{k} {v}' for k, v in reasons.items()) or 'none')
          + '. The map as reported with them left out, counted as not burnt, and counted as burnt:')
    display(bounds.round(3))
    display(area_est[['case', 'points', 'burnt_ha_from_labels', 'ci95_low_ha', 'ci95_high_ha']].round(0))
    print('\nLate windows: where the rule in Settings first calls each labelled point burnt, against the labels:')
    display(late_check)

### Caveats to carry into the report
- **Checked, not tuned.** At Checkpoint 3, thresholds fitted on half the labelled points did worse on the other half than the starting thresholds, so the starting ones stay. Accuracy comes from the labelled points, with `unsure` points reported both ways.
- **Unconfirmed late burns.** Burns first seen from 19 Nov are mapped as a class of their own and kept out of the headline figures. Damp soil under young wheat can pass all three tests, but some late burns may be real, so they are reported beside the headline, not deleted.
- **Short-lived scars.** Fields are often ploughed within days, so a burn between two cloudy views can be missed entirely. That makes burnt area an underestimate. In the label strips, a black field was back to the colour of tilled soil within two to three days.
- **Small and partial burns.** Burns covering only a small part of a 20 m pixel, such as straw burnt in small heaps, and burns of loose straw that leave the stubble standing, may never look dark enough to pass the rule. Such burning is outside the burnt area, and small, brief fires are rarely caught by fire counts either.
- **Clouds and smoke.** Smoke can be masked as cloud. The `clear_share_of_area` column shows how much of the area each window could see.
- **Generous matching.** The flagging window runs from the last clear view to the burn, and the lenient rules widen the distance. The missed shares are therefore conservative.
- **Edges.** Validation points come from pure 3 × 3 blocks of one map class, yet some still sit on a field or burn edge. Those are labelled `unsure` with the note `edge`, so accuracy at edges isn't measured.
- **Labels are interpretations.** They come from 10 m single-date imagery, not field visits, with a second, independent reading of every point. `unsure` points are reported both ways, not just dropped.
- **Boundary.** Sangrur is the pre-2021 district and includes Malerkotla.